# Stage 05 — Locked Source Test and External Validation

## Stage Overview
This stage performs the one locked BanglaBait TEST evaluation, platform-stratified BanglaClick evaluation, BaitBuster human-labelled replication, H1/H2 reliability analyses, primary H3 transport analysis, and H3-P controlled cue neutralization.

## Scientific Objective
H3 is primary. H1 and H2 are supporting reliability analyses. External datasets are inference/evaluation only; no target training, calibration, threshold fitting, cue selection, or policy tuning is permitted.


## Input Contract

### Required Input
One Kaggle Dataset containing the frozen Swarabyanjan external-data package.

The notebook automatically detects either a literal frozen ZIP or Kaggle's extracted frozen directory tree. Nested package wrappers are normalized automatically; no dataset-slug or path editing is required.

### Runtime
Kaggle Notebook.

### Accelerator
**GPU required** for the complete transformer inference and H3-P perturbation workload.

### Pipeline Position
03 / 04 / 05 / 06

### Upstream Requirements
Completed `PASS` outputs from Stage 03 and Stage 04, mounted as prior Kaggle notebook outputs or available under `/kaggle/working`.

### Read-only Inputs
Stage 03 model/clean TEST artifacts, Stage 04 temperature artifacts, BanglaClick, BaitBuster, and the discovered frozen package root are read-only.

### Generated Outputs
`/kaggle/working/03_results/05_external_validation/`, including frozen-input discovery metadata, immutable prediction artifacts, H1/H2/H3/H3-P results, sensitivity/statistical summaries, figures/tables, manifests, and hashes.

### Manual Uploads
No manually cleaned datasets, hand-generated intermediate files, or manual path configuration are required.


## Runtime Requirements
- Verified Stage 03 and Stage 04 outputs are required and read-only.
- The same frozen external-data package is required in ZIP or extracted-directory form.
- CUDA GPU is required for transformer external inference and H3-P.
- Internet is not required when all verified upstream model artifacts are mounted.
- No manual path, schema, threshold, or calibration editing is permitted.


## Environment Validation
The existing Kaggle compiled stack is preserved. Missing application packages are installed under constraints that prevent replacement of the core scientific stack, followed by a fresh-process import/API probe.


In [1]:
# Environment preflight — standard library only.
# The compiled Kaggle scientific stack is preserved. Only missing application-level packages may be installed.
from pathlib import Path
import importlib.metadata as _md
import importlib.util as _util
import json as _json
import os as _os
import subprocess as _subprocess
import sys as _sys
import tempfile as _tempfile
import textwrap as _textwrap

STAGE_ENVIRONMENT_ID = "SWARABYANJAN-KAGGLE-ENV-v2.0.0"
CORE_DISTRIBUTIONS = {
    "numpy":"numpy",
    "scipy":"scipy",
    "pandas":"pandas",
    "scikit-learn":"sklearn",
    "pyarrow":"pyarrow",
    "torch":"torch",
    "statsmodels":"statsmodels",
    "joblib":"joblib",
}
APPLICATION_REQUIREMENTS = [('interpret-core', 'interpret', 'interpret-core>=0.6,<0.8'), ('transformers', 'transformers', 'transformers>=4.45,<6'), ('sentencepiece', 'sentencepiece', 'sentencepiece>=0.1.99,<0.3'), ('openpyxl', 'openpyxl', 'openpyxl>=3.1,<4'), ('matplotlib', 'matplotlib', 'matplotlib>=3.7,<4')]

def _dist_version(name):
    try:
        return _md.version(name)
    except _md.PackageNotFoundError:
        return None

def _module_present(name):
    try:
        return _util.find_spec(name) is not None
    except Exception:
        return False

initial_versions = {name:_dist_version(name) for name in list(CORE_DISTRIBUTIONS)+[x[0] for x in APPLICATION_REQUIREMENTS]}
missing_core = [dist for dist,module in CORE_DISTRIBUTIONS.items() if not _module_present(module)]
if missing_core:
    raise RuntimeError(
        "WHAT FAILED: the standard Kaggle compiled scientific stack is incomplete. "
        f"Missing core distributions/modules: {missing_core}. "
        "WHY: installing or replacing NumPy/SciPy/pandas/scikit-learn/pyarrow/torch inside this notebook can create ABI conflicts. "
        "SAFE TO CONTINUE: NO. Start a fresh standard Kaggle Python runtime; do not upgrade the scientific stack."
    )

missing_apps = [(dist,module,spec) for dist,module,spec in APPLICATION_REQUIREMENTS if not _module_present(module)]
installed_now = []
if missing_apps:
    # Pin every existing compiled/core package so the dependency resolver cannot replace Kaggle's ABI-compatible stack.
    constraints = []
    for dist in CORE_DISTRIBUTIONS:
        v = _dist_version(dist)
        if v:
            constraints.append(f"{dist}=={v}")
    cfile = Path(_tempfile.gettempdir()) / f"swarabyanjan_05_constraints.txt"
    cfile.write_text("\n".join(constraints)+"\n", encoding="utf-8")
    specs = [spec for _,_,spec in missing_apps]
    cmd = [
        _sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--no-input", "--prefer-binary",
        "--constraint", str(cfile), *specs
    ]
    proc = _subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            "WHAT FAILED: installation of missing application-level dependencies failed. "
            f"Requested: {specs}. "
            "WHY: pip could not satisfy them without changing the pinned Kaggle scientific stack or network/package access failed. "
            "SAFE TO CONTINUE: NO.\n"
            f"pip stdout:\n{proc.stdout[-4000:]}\n"
            f"pip stderr:\n{proc.stderr[-4000:]}"
        )
    installed_now = specs

# Import/API validation occurs in a fresh subprocess before compiled packages are imported into this notebook kernel.
_probe = r"""
import json
import numpy, scipy, pandas, sklearn, pyarrow, torch, statsmodels, joblib, matplotlib, openpyxl
from statsmodels.stats.multitest import multipletests
from interpret.glassbox import ExplainableBoostingClassifier
from transformers import AutoTokenizer, AutoModelForSequenceClassification
print(json.dumps({
 "numpy":numpy.__version__,"scipy":scipy.__version__,"pandas":pandas.__version__,
 "sklearn":sklearn.__version__,"pyarrow":pyarrow.__version__,"torch":torch.__version__,
 "statsmodels":statsmodels.__version__,"matplotlib":matplotlib.__version__
}, sort_keys=True))
"""
probe_proc = _subprocess.run([_sys.executable, "-c", _probe], capture_output=True, text=True)
if probe_proc.returncode != 0:
    raise RuntimeError(
        "WHAT FAILED: post-setup import/API validation failed. "
        "WHY: the runtime is still incompatible or a required API is unavailable. "
        "SAFE TO CONTINUE: NO. Restart with a fresh Kaggle Python runtime rather than upgrading the scientific stack.\n"
        f"probe stdout:\n{probe_proc.stdout[-4000:]}\n"
        f"probe stderr:\n{probe_proc.stderr[-4000:]}"
    )

ENVIRONMENT_PREFLIGHT = {
    "schema_version": 1,
    "environment_id": STAGE_ENVIRONMENT_ID,
    "stage_id": "05",
    "policy": "preserve Kaggle compiled scientific stack; install only missing application-level packages",
    "accelerator_note": 'GPU required for Stage 05',
    "initial_versions": initial_versions,
    "installed_now": installed_now,
    "validated_versions": {name:_dist_version(name) for name in list(CORE_DISTRIBUTIONS)+[x[0] for x in APPLICATION_REQUIREMENTS]},
    "probe_stdout": probe_proc.stdout.strip(),
}
print("Environment validation: PASS")
print(probe_proc.stdout.strip())


Environment validation: PASS
{"matplotlib": "3.10.0", "numpy": "2.0.2", "pandas": "2.3.3", "pyarrow": "24.0.0", "scipy": "1.16.3", "sklearn": "1.6.1", "statsmodels": "0.14.6", "torch": "2.10.0+cu128"}


In [2]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import Counter, defaultdict
import hashlib, importlib.metadata as im, json, math, os, platform, random, re, shutil, subprocess, sys, unicodedata, zipfile
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.special import expit, logit
from scipy.stats import chi2
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss, f1_score
from statsmodels.stats.multitest import multipletests
import joblib

STAGE_ID="05"; STAGE_NAME="05_external_validation"; PROTOCOL_VERSION="SWARABYANJAN-CROSSPLATFORM-v1.0.0"
FREEZE_ID="EXTERNAL-DATASET-BYTES-v1"; FROZEN_ZIP_NAME="EXTERNAL-DATASET-BYTES-v1.zip"
EXPECTED_ZIP_SHA256="e0de5c224f406fe22d8ad30804217465f789c49c70e1d3afbe00afd28deb7ccf"
BANGLACLICK_COMMIT="2ba28763d8f80eaba7a649877eae1fbbcb78a0e7"
BANGLACLICK_RELATIVE_PATH="BanglaClick/repository_snapshot/Data/Final_4961_Dataset.xlsx"
BANGLACLICK_SHA256="2202f89445fa12f1880854edb50f10d37293bd006758fe1901c69213f058e060"; BANGLACLICK_SIZE=1277552
BAITBUSTER_SHA256="7f1bb757c9c5ee1a2922efcd45fb829b89e96d4465995e538158c4225f3c12c9"; BAITBUSTER_SIZE=175950911
SEEDS=[42,137,2718,31415,65537]; MODEL_KEYS=["ebm","tfidf_lr","banglabert","xlmr"]
BOOTSTRAP_REPS=5000; BOOTSTRAP_SEED=20260906; FDR_Q=0.05
WORK_ROOT=Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
INPUT_ROOT=Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
SEARCH_ROOTS=[INPUT_ROOT,WORK_ROOT,Path.cwd()]
OUT_ROOT=WORK_ROOT/"03_results"/STAGE_NAME; TEMP_ROOT=WORK_ROOT/f".{STAGE_NAME}_tmp"
if OUT_ROOT.exists() and any(OUT_ROOT.iterdir()): raise RuntimeError(f"Refusing overwrite: {OUT_ROOT}")
OUT_ROOT.mkdir(parents=True,exist_ok=True); TEMP_ROOT.mkdir(parents=True,exist_ok=True)

def utc_now(): return datetime.now(timezone.utc).isoformat()
def cjb(o): return json.dumps(o,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode("utf-8")
def sha256_file(p,chunk=8*1024*1024):
    h=hashlib.sha256(); f=open(p,"rb")
    while True:
        b=f.read(chunk)
        if not b: f.close(); return h.hexdigest()
        h.update(b)
def write_json(p,o): Path(p).write_text(json.dumps(o,ensure_ascii=False,sort_keys=True,indent=2)+"\n",encoding="utf-8",newline="\n")
def write_text(p,t): Path(p).write_text(t,encoding="utf-8",newline="\n")
def set_status(s,m,d=None): write_json(OUT_ROOT/"STAGE_STATUS.json",{"stage_id":STAGE_ID,"status":s,"message":m,"timestamp_utc":utc_now(),"details":d or {}}); print(f"STAGE 05 STATUS: {s} — {m}")
def hard_stop(m,d=None):
    set_status("HARD_STOP",m,d)
    if d:
        detail=json.dumps(d,ensure_ascii=False,sort_keys=True,default=str)
        raise RuntimeError(f"{m} | details={detail[:12000]}")
    raise RuntimeError(m)
def pkg(n):
    try:return im.version(n)
    except:return None
def git_state():
    try:
        head=subprocess.check_output(["git","rev-parse","HEAD"],stderr=subprocess.DEVNULL,text=True).strip()
        status=subprocess.check_output(["git","status","--porcelain"],stderr=subprocess.DEVNULL,text=True)
        return {"available":True,"head":head,"dirty":bool(status.strip()),"status_porcelain":status[:20000]}
    except Exception:
        return {"available":False,"head":None,"dirty":None,"status_porcelain":None}
write_json(OUT_ROOT/"ENVIRONMENT_PREFLIGHT.json",ENVIRONMENT_PREFLIGHT)
set_status("RUNNING","Stage-05 initialization complete")


STAGE 05 STATUS: RUNNING — Stage-05 initialization complete


## Dataset/Input Discovery
Stage 03/04 outputs and the frozen external package are recursively discovered, cryptographically verified, and duplicate-identical mounts are collapsed. BanglaClick sheet/column roles are selected from workbook/header structure only.

## Upstream and Frozen-Input Verification
Stage 03 and Stage 04 content identities are verified before use. The frozen external-data ZIP is rediscovered by pinned bytes; identical duplicate physical mounts are collapsed without accepting multiple distinct identities.


In [3]:
# ---------- Locate/verify Stage-03 and Stage-04 contracts ----------
def parse_sumfile(path):
    out={}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():continue
        h,rel=line.split(None,1);rel=rel.lstrip("*").strip().replace("\\","/")
        if not re.fullmatch(r"[0-9a-f]{64}",h.lower()):hard_stop("Malformed upstream checksum line",{"line":line})
        pp=Path(rel)
        if pp.is_absolute() or ".." in pp.parts:hard_stop("Unsafe upstream checksum path",{"path":rel})
        if rel in out:hard_stop("Duplicate upstream checksum path",{"path":rel})
        out[rel]=h.lower()
    return out

def find_unique(marker,checksum):
    c=[]
    for root in SEARCH_ROOTS:
        if not root.exists(): continue
        for p in root.rglob(marker):
            if (p.parent/checksum).is_file(): c.append(p.parent.resolve())
    c=sorted(set(c),key=str)
    if not c: hard_stop(f"Upstream artifact root not found for {marker}",{"safe_to_continue":False})
    identities={}
    for x in c:
        identities.setdefault(sha256_file(x/checksum),[]).append(x)
    if len(identities)!=1:
        hard_stop(f"Multiple distinct upstream identities found for {marker}",{"candidates":[str(x) for x in c],"safe_to_continue":False})
    chosen=sorted(next(iter(identities.values())),key=str)[0]
    if len(c)>1: print("Upstream discovery: identical physical mounts collapsed; using",chosen)
    return chosen
def verify_root(root,sumname):
    sums=parse_sumfile(root/sumname);fail=[]
    for rel,h in sorted(sums.items()):
        p=root/rel
        if not p.is_file():fail.append((rel,"missing"))
        elif sha256_file(p)!=h:fail.append((rel,"sha256_mismatch"))
    if fail:hard_stop("Upstream artifact verification failed",{"root":str(root),"failures":fail[:20],"count":len(fail)})
    return sha256_file(root/sumname)

S3=find_unique("MODEL_ARTIFACT_MANIFEST.json","STAGE03_SHA256SUMS.txt"); S4=find_unique("CALIBRATION_ARTIFACT_MANIFEST.json","STAGE04_SHA256SUMS.txt")
S3_SUM=verify_root(S3,"STAGE03_SHA256SUMS.txt"); S4_SUM=verify_root(S4,"STAGE04_SHA256SUMS.txt")
MODEL_MAN=json.loads((S3/"MODEL_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8")); CONFIG=json.loads((S3/"MODEL_CONFIG_LOCK.json").read_text(encoding="utf-8")); CUE_SCHEMA=json.loads((S3/"CUE_FEATURE_SCHEMA.json").read_text(encoding="utf-8"))
CAL_MAN=json.loads((S4/"CALIBRATION_ARTIFACT_MANIFEST.json").read_text(encoding="utf-8")); TEMPS=json.loads((S4/"TEMPERATURES.json").read_text(encoding="utf-8"))["temperatures"]
if CAL_MAN["upstream_stage03_checksum_manifest_sha256"]!=S3_SUM: hard_stop("Stage-04 does not point to this exact Stage-03 checksum manifest")
if MODEL_MAN["frozen_package_sha256"]!=EXPECTED_ZIP_SHA256:hard_stop("Frozen package identity mismatch in Stage-03 manifest")
if sorted(MODEL_MAN["seeds"])!=sorted(SEEDS):hard_stop("Seed contract mismatch")
if sorted({r["model"] for r in MODEL_MAN["models"]})!=sorted(MODEL_KEYS):hard_stop("Model-panel contract mismatch")
if CONFIG.get("config_lock_sha256")!=MODEL_MAN.get("model_config_lock_sha256"):hard_stop("Model-config lock mismatch")
if CUE_SCHEMA.get("cue_schema_sha256")!=MODEL_MAN.get("cue_schema_sha256"):hard_stop("Cue-schema lock mismatch")
expected_seed_keys={str(x) for x in SEEDS}
for m in MODEL_KEYS:
    if m not in TEMPS or set(TEMPS[m])!=expected_seed_keys:hard_stop("Temperature artifact seed/model contract mismatch",{"model":m})

S3_RUN=json.loads((S3/"RUN_METADATA.json").read_text(encoding="utf-8"))
s3_pkgs=S3_RUN.get("environment",{}).get("packages",{})
critical_serialization_versions={"scikit-learn":pkg("scikit-learn"),"interpret-core":pkg("interpret-core"),"joblib":pkg("joblib")}
version_mismatch={}
for name,current in critical_serialization_versions.items():
    expected=s3_pkgs.get(name)
    if expected and current and expected!=current:version_mismatch[name]={"stage03":expected,"current":current}
if version_mismatch:
    hard_stop("Critical serialized-model library versions differ from Stage 03",
              {"mismatches":version_mismatch,"safe_to_continue":False,
               "action":"Run Stage 05 on the same Kaggle base image/package set as Stage 03; do not upgrade the compiled stack."})
print("Upstream contracts verified")


Upstream contracts verified


In [4]:

# ---------- Frozen Stage-01 package discovery and verification ----------
# Supports both a literal pinned ZIP and Kaggle's automatically extracted directory tree.
FROZEN_REQUIRED_CONTROLS = (
    "ACQUISITION_MANIFEST.json",
    "ACQUISITION_STATUS.json",
    "SOURCES_LOCK.json",
    "SHA256SUMS.txt",
    "README_HANDOFF.txt",
)
FROZEN_REQUIRED_DATASET_DIRS = (
    "BaitBuster-Bangla",
    "BanFakeNews-2.0",
    "BanMANI",
    "BanglaBait",
    "BanglaClick",
)

class FrozenPackageDiscoveryError(RuntimeError):
    pass


def safe_extract_zip(zip_path, destination):
    zip_path = Path(zip_path)
    destination = Path(destination).resolve()
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            target = (destination / info.filename).resolve()
            if target != destination and destination not in target.parents:
                raise FrozenPackageDiscoveryError(f"Unsafe ZIP member: {info.filename}")
        bad = zf.testzip()
        if bad is not None:
            raise FrozenPackageDiscoveryError(f"ZIP CRC verification failed at member: {bad}")
        zf.extractall(destination)


def parse_sha256sums(path):
    out = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        parts = line.split(None, 1)
        if len(parts) != 2 or not re.fullmatch(r"[0-9a-fA-F]{64}", parts[0]):
            raise FrozenPackageDiscoveryError(f"Malformed SHA256SUMS.txt line: {line!r}")
        rel = parts[1].lstrip("*").strip().replace("\\", "/")
        pp = PurePosixPath(rel)
        if pp.is_absolute() or ".." in pp.parts:
            raise FrozenPackageDiscoveryError(f"Unsafe path in SHA256SUMS.txt: {rel}")
        if rel in out:
            raise FrozenPackageDiscoveryError(f"Duplicate path in SHA256SUMS.txt: {rel}")
        out[rel] = parts[0].lower()
    if not out:
        raise FrozenPackageDiscoveryError("SHA256SUMS.txt is empty")
    return out


def _checksum_target_variants(root, rel):
    """Return safe candidate paths, tolerating one or more repeated package-name prefixes."""
    root = Path(root).resolve()
    pp = PurePosixPath(rel)
    parts = list(pp.parts)
    variants = [parts[:]]
    stripped = parts[:]
    while stripped and stripped[0] in {root.name, FREEZE_ID}:
        stripped = stripped[1:]
        if stripped:
            variants.append(stripped[:])
    out = []
    seen = set()
    for vp in variants:
        p = root.joinpath(*vp).resolve()
        if p != root and root not in p.parents:
            continue
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out



def require_declared_frozen_file(root, checksums, target, role):
    """Require a consumed scientific input to be uniquely declared and checksum-matched."""
    root = Path(root).resolve()
    target = Path(target).resolve()
    hits = []
    for rel, expected in checksums.items():
        for candidate in _checksum_target_variants(root, rel):
            if candidate.resolve() == target:
                hits.append((rel, expected))
    # Repeated wrapper-normalization may generate duplicate references to the same declaration.
    hits = sorted(set(hits))
    if len(hits) != 1:
        hard_stop(
            "Consumed frozen input is not uniquely declared in SHA256SUMS.txt",
            {"role": role, "path": str(target), "declarations": [x[0] for x in hits], "safe_to_continue": False},
        )
    rel, expected = hits[0]
    if not target.is_file():
        hard_stop("Declared frozen input is missing", {"role": role, "path": str(target), "safe_to_continue": False})
    actual = sha256_file(target)
    if actual != expected:
        hard_stop(
            "Consumed frozen input checksum mismatch",
            {"role": role, "path": str(target), "declared_path": rel, "expected": expected, "actual": actual, "safe_to_continue": False},
        )
    return {"role": role, "path": str(target), "declared_path": rel, "sha256": actual}

def validate_frozen_root(root):
    """Validate one extracted package root and return a content-derived package identity."""
    root = Path(root).resolve()
    missing_controls = [name for name in FROZEN_REQUIRED_CONTROLS if not (root / name).is_file()]
    missing_dirs = [name for name in FROZEN_REQUIRED_DATASET_DIRS if not (root / name).is_dir()]
    if missing_controls or missing_dirs:
        raise FrozenPackageDiscoveryError(
            f"Structural markers missing at {root}: controls={missing_controls}, dataset_dirs={missing_dirs}"
        )

    checksums = parse_sha256sums(root / "SHA256SUMS.txt")
    failures = []
    for rel, expected in sorted(checksums.items()):
        candidates = [p for p in _checksum_target_variants(root, rel) if p.is_file()]
        if not candidates:
            failures.append({"path": rel, "reason": "missing"})
            continue
        # Repeated wrapper-prefix variants may resolve to more than one existing path only in malformed trees.
        actuals = [(p, sha256_file(p)) for p in candidates]
        matches = [(p, h) for p, h in actuals if h == expected]
        if not matches:
            failures.append({
                "path": rel, "reason": "sha256_mismatch",
                "expected": expected, "actuals": [(str(p), h) for p, h in actuals]
            })
    if failures:
        raise FrozenPackageDiscoveryError(
            f"File-level checksum verification failed at {root}: {failures[:10]}"
        )

    control_hashes = {name: sha256_file(root / name) for name in FROZEN_REQUIRED_CONTROLS}
    identity_payload = {
        "schema_version": 1,
        "freeze_id": FREEZE_ID,
        "controls": control_hashes,
        "declared_sha256sums": sorted(checksums.items()),
        "dataset_dirs": sorted(FROZEN_REQUIRED_DATASET_DIRS),
    }
    package_identity = hashlib.sha256(
        json.dumps(identity_payload, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()
    return {
        "root": root,
        "package_identity": package_identity,
        "checksum_entry_count": len(checksums),
        "control_hashes": control_hashes,
        "checksums": checksums,
    }


def _find_structural_roots(search_roots, scan_warnings=None):
    candidates = set()
    scan_warnings = [] if scan_warnings is None else scan_warnings
    for search_root in search_roots:
        search_root = Path(search_root)
        if not search_root.exists():
            continue
        try:
            markers = search_root.rglob("SOURCES_LOCK.json")
            for marker in markers:
                root = marker.parent.resolve()
                if all((root / name).is_file() for name in FROZEN_REQUIRED_CONTROLS[:4]):
                    candidates.add(root)
        except (OSError, PermissionError) as exc:
            scan_warnings.append({
                "operation": "recursive_control_manifest_scan",
                "path": str(search_root),
                "error": f"{type(exc).__name__}: {exc}",
            })
    return sorted(candidates, key=str)


def discover_frozen_package(search_roots, temp_root, expected_zip_sha256=EXPECTED_ZIP_SHA256,
                            zip_name=FROZEN_ZIP_NAME):
    """
    Discover a canonical frozen package in either mode:
      ZIP: literal EXTERNAL-DATASET-BYTES-v1.zip with the pinned ZIP SHA.
      EXTRACTED_DIRECTORY: structurally valid extracted package root.

    Identical content identities are deduplicated. Multiple distinct valid identities hard-stop.
    """
    search_roots = [Path(p) for p in search_roots if Path(p).exists()]
    temp_root = Path(temp_root)
    temp_root.mkdir(parents=True, exist_ok=True)

    valid = []
    invalid = []
    scan_warnings = []

    # Mode B first: directly mounted/extracted Kaggle package tree.
    for root in _find_structural_roots(search_roots, scan_warnings):
        try:
            meta = validate_frozen_root(root)
            valid.append({
                "mode": "EXTRACTED_DIRECTORY",
                "source_path": root,
                **meta,
                "zip_sha256": None,
            })
        except Exception as exc:
            invalid.append({"mode": "EXTRACTED_DIRECTORY", "path": str(root), "error": f"{type(exc).__name__}: {exc}"})

    # Mode A: literal canonical ZIP, if mounted.
    zip_paths = set()
    for search_root in search_roots:
        try:
            for p in search_root.rglob(zip_name):
                if p.is_file():
                    zip_paths.add(p.resolve())
        except (OSError, PermissionError) as exc:
            scan_warnings.append({
                "operation": "recursive_zip_scan",
                "path": str(search_root),
                "error": f"{type(exc).__name__}: {exc}",
            })

    for idx, zp in enumerate(sorted(zip_paths, key=str)):
        try:
            zhash = sha256_file(zp)
            if zhash != expected_zip_sha256:
                raise FrozenPackageDiscoveryError(
                    f"ZIP SHA-256 mismatch for {zp}: expected {expected_zip_sha256}, observed {zhash}"
                )
            extract_dir = temp_root / f"zip_{idx:02d}_{zhash[:12]}"
            safe_extract_zip(zp, extract_dir)
            local_scan_warnings = []
            roots = _find_structural_roots([extract_dir], local_scan_warnings)
            scan_warnings.extend(local_scan_warnings)
            if not roots:
                raise FrozenPackageDiscoveryError("ZIP contains no structurally valid frozen package root")
            local_valid = []
            local_rejections = []
            for eroot in roots:
                try:
                    local_valid.append(validate_frozen_root(eroot))
                except Exception as exc:
                    local_rejections.append({"path": str(eroot), "error": f"{type(exc).__name__}: {exc}"})
            if len(local_valid) != 1:
                raise FrozenPackageDiscoveryError(
                    f"ZIP must resolve to exactly one valid package root; found {len(local_valid)}; "
                    f"rejected candidates: {local_rejections[:20]}"
                )
            meta = local_valid[0]
            valid.append({
                "mode": "ZIP",
                "source_path": zp,
                **meta,
                "zip_sha256": zhash,
            })
        except Exception as exc:
            invalid.append({"mode": "ZIP", "path": str(zp), "error": f"{type(exc).__name__}: {exc}"})

    if not valid:
        raise FrozenPackageDiscoveryError(
            "No valid frozen Swarabyanjan package was found. "
            "Accepted inputs are either the literal pinned ZIP or an extracted directory containing "
            "the Stage-01 control manifests and expected dataset directories. "
            f"Rejected candidates: {invalid[:20]}"
        )

    by_identity = defaultdict(list)
    for item in valid:
        by_identity[item["package_identity"]].append(item)

    if len(by_identity) > 1:
        conflict = {
            identity: [
                {"mode": x["mode"], "source_path": str(x["source_path"]), "root": str(x["root"])}
                for x in items
            ]
            for identity, items in sorted(by_identity.items())
        }
        raise FrozenPackageDiscoveryError(
            "Multiple genuinely conflicting frozen package identities were discovered: "
            + json.dumps(conflict, ensure_ascii=False, sort_keys=True)
        )

    identity, identical = next(iter(by_identity.items()))
    # Prefer a read-only extracted Kaggle tree when both representations contain identical bytes.
    identical = sorted(
        identical,
        key=lambda x: (0 if x["mode"] == "EXTRACTED_DIRECTORY" else 1, str(x["source_path"]))
    )
    selected = identical[0]
    selected = dict(selected)
    selected["duplicate_equivalent_candidates"] = [
        {"mode": x["mode"], "source_path": str(x["source_path"]), "root": str(x["root"])}
        for x in identical[1:]
    ]
    selected["invalid_candidates"] = invalid
    selected["scan_warnings"] = scan_warnings
    return selected


search_roots = [INPUT_ROOT]
if Path.cwd().resolve() != INPUT_ROOT.resolve():
    search_roots.append(Path.cwd())

try:
    FROZEN_DISCOVERY = discover_frozen_package(
        search_roots=search_roots,
        temp_root=TEMP_ROOT / "frozen_input_discovery",
    )
except FrozenPackageDiscoveryError as exc:
    hard_stop(
        "Frozen external-data package discovery failed",
        {
            "what_failed": str(exc),
            "why": "Neither a valid pinned ZIP nor a structurally/checksum-valid extracted frozen package could be selected unambiguously.",
            "safe_to_continue": False,
        },
    )

FROZEN_ROOT = Path(FROZEN_DISCOVERY["root"]).resolve()
FROZEN_INPUT_MODE = FROZEN_DISCOVERY["mode"]
FROZEN_INPUT_SOURCE = Path(FROZEN_DISCOVERY["source_path"]).resolve()
FROZEN_PACKAGE_IDENTITY = FROZEN_DISCOVERY["package_identity"]

SOURCES_LOCK = json.loads((FROZEN_ROOT / "SOURCES_LOCK.json").read_text(encoding="utf-8"))
lock_text = json.dumps(SOURCES_LOCK, ensure_ascii=False, sort_keys=True)
for token in [
    "ff22fb3b423952959a80dc5d9c6c3345b21cd145",
    "2ba28763d8f80eaba7a649877eae1fbbcb78a0e7",
    "be1528ba40dd8828c5c06ee70b7574cca22b8ee1",
]:
    if token not in lock_text:
        hard_stop("Frozen source-identity token absent from SOURCES_LOCK.json", {"token": token})

BANGLACLICK_PATH = FROZEN_ROOT / BANGLACLICK_RELATIVE_PATH
BAITBUSTER_PATH = FROZEN_ROOT / "BaitBuster-Bangla/Mendeley_V3_public_author_serializations/BaitBuster-Bangla_253070_18c_HL10k_AIL.parquet"
if not BANGLACLICK_PATH.is_file():
    hard_stop("BanglaClick canonical file missing", {"path": str(BANGLACLICK_PATH)})
if BANGLACLICK_PATH.stat().st_size != BANGLACLICK_SIZE or sha256_file(BANGLACLICK_PATH) != BANGLACLICK_SHA256:
    hard_stop("BanglaClick pinned workbook identity mismatch", {
        "path": str(BANGLACLICK_PATH), "expected_size": BANGLACLICK_SIZE,
        "observed_size": BANGLACLICK_PATH.stat().st_size,
        "expected_sha256": BANGLACLICK_SHA256, "observed_sha256": sha256_file(BANGLACLICK_PATH),
        "safe_to_continue": False,
    })
if not BAITBUSTER_PATH.is_file():
    hard_stop("BaitBuster canonical Parquet missing", {"path": str(BAITBUSTER_PATH)})
if BAITBUSTER_PATH.stat().st_size != BAITBUSTER_SIZE or sha256_file(BAITBUSTER_PATH) != BAITBUSTER_SHA256:
    hard_stop("BaitBuster pinned identity mismatch")

CONSUMED_EXTERNAL_CHECKSUM_DECLARATIONS = {
    "banglaclick": require_declared_frozen_file(FROZEN_ROOT, FROZEN_DISCOVERY["checksums"], BANGLACLICK_PATH, "BanglaClick workbook"),
    "baitbuster": require_declared_frozen_file(FROZEN_ROOT, FROZEN_DISCOVERY["checksums"], BAITBUSTER_PATH, "BaitBuster canonical Parquet"),
}

write_json(OUT_ROOT / "FROZEN_INPUT_DISCOVERY.json", {
    "schema_version": 1,
    "mode": FROZEN_INPUT_MODE,
    "source_path": str(FROZEN_INPUT_SOURCE),
    "root": str(FROZEN_ROOT),
    "content_identity_sha256": FROZEN_PACKAGE_IDENTITY,
    "canonical_zip_sha256_provenance": EXPECTED_ZIP_SHA256,
    "observed_zip_sha256": FROZEN_DISCOVERY.get("zip_sha256"),
    "duplicate_equivalent_candidates": FROZEN_DISCOVERY.get("duplicate_equivalent_candidates", []),
    "invalid_candidates": FROZEN_DISCOVERY.get("invalid_candidates", []),
    "scan_warnings": FROZEN_DISCOVERY.get("scan_warnings", []),
    "consumed_input_checksum_declarations": CONSUMED_EXTERNAL_CHECKSUM_DECLARATIONS,
})
print("External frozen identities: PASS —", f"mode={FROZEN_INPUT_MODE}", f"root={FROZEN_ROOT}")


External frozen identities: PASS — mode=EXTRACTED_DIRECTORY root=/kaggle/input/datasets/smalakarishere/swarabyanjan-frozen-dataset-v1/EXTERNAL-DATASET-BYTES-v1/EXTERNAL-DATASET-BYTES-v1


## Dataset Loading and Semantic Adapters
BanglaClick remains the primary external platform-stratified benchmark. BaitBuster uses only the human-labelled subset with `title_debiased` and `channel_id` clustering. Missing text, missing platform strata, unharmonizable labels, and single-class platform strata hard-stop before inference.


In [5]:
# ---------- Locked normalization/cue extraction ----------
def norm_text(v):
    if v is None or (isinstance(v,float) and math.isnan(v)):return ""
    t=unicodedata.normalize("NFKC",str(v));t=t.replace("\u200b","").replace("\u200c","").replace("\u200d","").replace("\ufeff","")
    return re.sub(r"\s+"," ",t.casefold().strip())
def sha_text(t):return hashlib.sha256(str(t).encode("utf-8")).hexdigest()
def term_count(text,terms):
    cc=Counter(text.split());return sum(text.count(norm_text(t)) if " " in norm_text(t) else cc.get(norm_text(t),0) for t in terms)
def cue_features(titles):
    rows=[]
    for raw in titles:
        text=norm_text(raw);toks=text.split();r={}
        for name,spec in CUE_SCHEMA["families"].items():
            if name=="specificity_numerals":c=sum(len(re.findall(p,text,flags=re.I)) for p in spec["patterns"])
            elif name=="punctuation_intensity":c=sum(text.count(p) for p in spec["punctuation"])+len(re.findall(spec["repeated_pattern"],text))
            else:c=term_count(text,spec.get("terms",[]))+sum(text.count(p) for p in spec.get("punctuation",[]))
            r[name]=int(c)
        chars=len(text);latin=sum("a"<=ch<="z" for ch in text)
        r.update({"token_count":len(toks),"character_count":chars,"mean_token_length":float(np.mean([len(x) for x in toks])) if toks else 0.0,"latin_code_mixing_proportion":latin/chars if chars else 0.0})
        rows.append(r)
    return pd.DataFrame(rows,columns=list(CUE_SCHEMA["families"])+CUE_SCHEMA["structural_controls"])
CUE_FAMILIES=list(CUE_SCHEMA["families"])


## BanglaClick Data Schema

The canonical BanglaClick workbook is frozen to author commit `2ba28763d8f80eaba7a649877eae1fbbcb78a0e7` and repository-relative path `BanglaClick/repository_snapshot/Data/Final_4961_Dataset.xlsx`.

The physical workbook contract is exact and fail-closed:

- worksheet: `Sheet1`
- canonical teaser text: `TEXT`
- canonical source/platform field: `SOURCE`
- canonical multi-level clickbait field: `FINAL_CLICKBAIT_LEVEL`
- diagnostic fields: `FAST_FAIL_COUNT`, `FAST_LEVEL`, `STRONG_TRIGGER_COUNT`, `WEAK_TRIGGER_COUNT`, `TRIGGER_LEVEL`

The notebook verifies the complete 14-column header in order, harmless normalized headers, absence of normalized-header collisions, workbook byte identity, required-field usability, source/platform value handling, and the frozen 0-versus-positive label harmonization. No target label distribution, prediction, prevalence, correlation, or performance metric is used to select a schema role.

For the locked four-platform evaluation, `SOURCE` is interpreted deterministically: `News site` → `news`, `Youtube` → `youtube`, and `Social Media` is disambiguated only by the structural `POST_URL` host (`facebook.com` or `twitter.com`). `SOURCE_URL` is retained as the source-entity cluster identifier, with no target-model information involved.

The physical schema audit is persisted to `BANGLACLICK_SCHEMA_PROBE.json` and the accepted machine-readable contract/data audit to `BANGLACLICK_SCHEMA_REPORT.json`.


In [6]:
# ---------- Strict external-data adapters; no learned harmonization ----------
from openpyxl import load_workbook
from urllib.parse import urlparse

def ncol(x):
    return re.sub(r"[^a-z0-9]+","_",unicodedata.normalize("NFKC",str(x)).casefold()).strip("_")

BANGLACLICK_EXPECTED_SHEET = "Sheet1"
BANGLACLICK_EXPECTED_HEADERS = [
    "SOURCE",
    "SOURCE_URL",
    "POST_URL",
    "TEXT",
    "CONTENT",
    "CONTENT_URL",
    "MEDIA_TYPE",
    "MEDIA_LINK",
    "FAST_FAIL_COUNT",
    "FAST_LEVEL",
    "STRONG_TRIGGER_COUNT",
    "WEAK_TRIGGER_COUNT",
    "TRIGGER_LEVEL",
    "FINAL_CLICKBAIT_LEVEL",
]
BANGLACLICK_TEXT_COLUMN = "TEXT"
BANGLACLICK_PLATFORM_COLUMN = "SOURCE"
BANGLACLICK_SCORE_COLUMN = "FINAL_CLICKBAIT_LEVEL"
BANGLACLICK_DIAGNOSTIC_COLUMNS = [
    "FAST_FAIL_COUNT",
    "FAST_LEVEL",
    "STRONG_TRIGGER_COUNT",
    "WEAK_TRIGGER_COUNT",
    "TRIGGER_LEVEL",
]
BANGLACLICK_SOURCE_ENTITY_COLUMN = "SOURCE_URL"
BANGLACLICK_POST_URL_COLUMN = "POST_URL"
BANGLACLICK_ALLOWED_RAW_SOURCES = {"News site", "Social Media", "Youtube"}
BANGLACLICK_ALLOWED_LABEL_LEVELS = {0, 1, 2, 3}
BANGLACLICK_REQUIRED_PLATFORMS = {"news", "facebook", "youtube", "twitter"}

BANGLACLICK_SCHEMA_CONTRACT = {
    "schema_version": 3,
    "selection_basis": "exact physical workbook identity + exact sheet/header identity only",
    "source_commit": BANGLACLICK_COMMIT,
    "repository_relative_path": BANGLACLICK_RELATIVE_PATH,
    "workbook_sha256": BANGLACLICK_SHA256,
    "workbook_size": BANGLACLICK_SIZE,
    "sheet_name": BANGLACLICK_EXPECTED_SHEET,
    "raw_headers": BANGLACLICK_EXPECTED_HEADERS,
    "normalized_headers": [ncol(x) for x in BANGLACLICK_EXPECTED_HEADERS],
    "text_column": BANGLACLICK_TEXT_COLUMN,
    "platform_column": BANGLACLICK_PLATFORM_COLUMN,
    "score_column": BANGLACLICK_SCORE_COLUMN,
    "diagnostic_columns": BANGLACLICK_DIAGNOSTIC_COLUMNS,
    "source_entity_column": BANGLACLICK_SOURCE_ENTITY_COLUMN,
    "platform_value_rule": {
        "News site": "news",
        "Youtube": "youtube",
        "Social Media": "POST_URL host: facebook.com -> facebook; twitter.com -> twitter",
    },
    "binary_label_rule": {
        "0": 0,
        "1": 1,
        "2": 1,
        "3": 1,
        "meaning": "frozen Stage-05 zero-vs-positive harmonization; no target fitting",
    },
}

def _blank_nonmissing_count(series):
    return int(series.dropna().astype(str).map(lambda x: x.strip() == "").sum())

def _normalized_header_map(raw_headers):
    out = defaultdict(list)
    for raw in raw_headers:
        out[ncol(raw)].append(str(raw))
    return out

def _safe_json_value(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    return v

def _url_host(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if not s:
        return None
    try:
        host = urlparse(s).netloc.casefold()
    except Exception:
        return None
    if host.startswith("www."):
        host = host[4:]
    return host or None

def parse_banglaclick_platform(source_value, post_url_value):
    if pd.isna(source_value):
        return None
    source = str(source_value).strip()
    if source == "News site":
        return "news"
    if source == "Youtube":
        return "youtube"
    if source == "Social Media":
        host = _url_host(post_url_value)
        if host == "facebook.com":
            return "facebook"
        if host == "twitter.com":
            return "twitter"
        return None
    return None

def harmonize_multilevel_clickbait(v):
    if pd.isna(v) or isinstance(v, (bool, np.bool_)):
        return None
    if isinstance(v, (int, float, np.integer, np.floating)):
        fv = float(v)
        if not np.isfinite(fv) or not fv.is_integer():
            return None
        level = int(fv)
        if level not in BANGLACLICK_ALLOWED_LABEL_LEVELS:
            return None
        return 0 if level == 0 else 1
    return None

def harmonize_baitbuster(v):
    if pd.isna(v):return None
    if isinstance(v,(bool,np.bool_)):return int(v)
    if isinstance(v,(int,float,np.integer,np.floating)) and float(v) in {0.0,1.0}:return int(v)
    s=norm_text(v)
    if s in {"0","not clickbait","not_clickbait","non clickbait","non-clickbait","no","false"}:return 0
    if s in {"1","clickbait","yes","true"}:return 1
    return None

def inspect_banglaclick_workbook(path):
    """
    Inspect the physical workbook header without pandas header mangling.
    Schema acceptance is exact and pre-specified; values are recorded only after roles are frozen.
    """
    path = Path(path)
    if not path.is_file():
        hard_stop("BanglaClick canonical file missing", {"path": str(path), "safe_to_continue": False})

    observed_size = int(path.stat().st_size)
    observed_sha = sha256_file(path)
    if observed_size != BANGLACLICK_SIZE or observed_sha != BANGLACLICK_SHA256:
        hard_stop("BanglaClick physical workbook identity mismatch", {
            "expected_size": BANGLACLICK_SIZE, "observed_size": observed_size,
            "expected_sha256": BANGLACLICK_SHA256, "observed_sha256": observed_sha,
            "safe_to_continue": False,
        })

    wb = load_workbook(path, read_only=True, data_only=False)
    sheet_names = [str(x) for x in wb.sheetnames]
    if sheet_names != [BANGLACLICK_EXPECTED_SHEET]:
        hard_stop("BanglaClick workbook sheet contract mismatch", {
            "expected_sheet_names": [BANGLACLICK_EXPECTED_SHEET],
            "observed_sheet_names": sheet_names,
            "safe_to_continue": False,
        })

    ws = wb[BANGLACLICK_EXPECTED_SHEET]
    header_row = next(ws.iter_rows(min_row=1, max_row=1, values_only=True))
    raw_headers = ["" if x is None else str(x) for x in header_row]
    normalized_headers = [ncol(x) for x in raw_headers]
    norm_map = _normalized_header_map(raw_headers)
    duplicate_normalized = {k: v for k, v in sorted(norm_map.items()) if len(v) > 1}

    if duplicate_normalized:
        hard_stop("BanglaClick normalized-header collision", {
            "duplicate_normalized_headers": duplicate_normalized,
            "safe_to_continue": False,
        })
    if raw_headers != BANGLACLICK_EXPECTED_HEADERS:
        hard_stop("BanglaClick exact header contract mismatch", {
            "expected_headers": BANGLACLICK_EXPECTED_HEADERS,
            "observed_headers": raw_headers,
            "safe_to_continue": False,
        })

    probe = {
        "schema_version": 3,
        "path": str(path),
        "filename": path.name,
        "repository_relative_path": BANGLACLICK_RELATIVE_PATH,
        "source_commit": BANGLACLICK_COMMIT,
        "workbook_size": observed_size,
        "workbook_sha256": observed_sha,
        "workbook_sheet_names": sheet_names,
        "sheet_name": BANGLACLICK_EXPECTED_SHEET,
        "excel_max_row_including_header": int(ws.max_row),
        "column_count": int(ws.max_column),
        "raw_headers": raw_headers,
        "normalized_headers": normalized_headers,
        "duplicate_normalized_headers": duplicate_normalized,
        "duplicate_normalized_header_status": "NONE",
        "frozen_role_mapping": {
            "text": BANGLACLICK_TEXT_COLUMN,
            "platform_source": BANGLACLICK_PLATFORM_COLUMN,
            "score_label": BANGLACLICK_SCORE_COLUMN,
            "diagnostics": BANGLACLICK_DIAGNOSTIC_COLUMNS,
            "source_entity_for_clustering": BANGLACLICK_SOURCE_ENTITY_COLUMN,
        },
        "role_selection_inputs": "exact sheet/header identity and pinned workbook bytes only; no label distribution, prevalence, model output, correlation, or performance input",
    }
    wb.close()
    write_json(OUT_ROOT / "BANGLACLICK_SCHEMA_PROBE.json", probe)
    print("BanglaClick exact physical schema: PASS")
    print(json.dumps(probe, ensure_ascii=False, indent=2))
    return probe

def _validate_diagnostic_columns(bc_raw):
    diagnostics = {}
    for col in BANGLACLICK_DIAGNOSTIC_COLUMNS:
        s = bc_raw[col]
        rec = {
            "dtype": str(s.dtype),
            "missing_count": int(s.isna().sum()),
            "blank_nonmissing_count": _blank_nonmissing_count(s),
            "unique_nonmissing_count": int(s.nunique(dropna=True)),
        }

        if col == "FAST_FAIL_COUNT":
            parsed = s.dropna().astype(str).str.extract(r"^\s*(\d+)", expand=False)
            invalid = s.dropna()[parsed.isna()]
            if len(invalid):
                hard_stop("BanglaClick FAST_FAIL_COUNT contains an unparseable diagnostic value", {
                    "values": sorted(map(str, invalid.unique())),
                    "safe_to_continue": False,
                })
            observed = sorted({int(x) for x in parsed.astype(int).unique()})
            if not set(observed).issubset({0,1,2,3,4}):
                hard_stop("BanglaClick FAST_FAIL_COUNT leading count is outside the frozen diagnostic range", {
                    "observed_leading_counts": observed, "safe_to_continue": False,
                })
            rec["observed_raw_values"] = sorted(map(str, s.dropna().unique()))
            rec["parsed_leading_count_values"] = observed
        else:
            num = pd.to_numeric(s, errors="coerce")
            invalid_mask = s.notna() & num.isna()
            if invalid_mask.any():
                hard_stop("BanglaClick numeric diagnostic contains a non-numeric value", {
                    "column": col,
                    "values": sorted(map(str, s[invalid_mask].unique())),
                    "safe_to_continue": False,
                })
            nonmissing = num.dropna()
            if ((nonmissing % 1) != 0).any() or (nonmissing < 0).any():
                hard_stop("BanglaClick numeric diagnostic contains a non-integer/negative value", {
                    "column": col, "safe_to_continue": False,
                })
            rec["observed_numeric_values"] = sorted(int(x) for x in nonmissing.astype(int).unique())
        diagnostics[col] = rec
    return diagnostics

def load_banglaclick_frozen(path):
    probe = inspect_banglaclick_workbook(path)
    bc_raw = pd.read_excel(path, sheet_name=BANGLACLICK_EXPECTED_SHEET, engine="openpyxl")

    observed_headers = [str(x) for x in bc_raw.columns]
    if observed_headers != BANGLACLICK_EXPECTED_HEADERS:
        hard_stop("BanglaClick pandas load changed the exact header contract", {
            "expected_headers": BANGLACLICK_EXPECTED_HEADERS,
            "observed_headers": observed_headers,
            "safe_to_continue": False,
        })

    # Core roles are frozen by exact physical header identity before any value inspection.
    bc_text = BANGLACLICK_TEXT_COLUMN
    bc_platform = BANGLACLICK_PLATFORM_COLUMN
    bc_score = BANGLACLICK_SCORE_COLUMN
    diag_cols = {ncol(c): c for c in BANGLACLICK_DIAGNOSTIC_COLUMNS}
    source_entity_col = BANGLACLICK_SOURCE_ENTITY_COLUMN

    missingness = {}
    for col in BANGLACLICK_EXPECTED_HEADERS:
        s = bc_raw[col]
        missingness[col] = {
            "missing_count": int(s.isna().sum()),
            "blank_nonmissing_count": _blank_nonmissing_count(s),
        }

    if bc_raw[bc_text].isna().any() or bc_raw[bc_text].map(norm_text).eq("").any():
        hard_stop("BanglaClick contains missing/empty primary text", {
            "sheet": BANGLACLICK_EXPECTED_SHEET,
            "text_column": bc_text,
            "safe_to_continue": False,
        })
    if bc_raw[bc_platform].isna().any() or bc_raw[bc_platform].astype(str).map(str.strip).eq("").any():
        hard_stop("BanglaClick contains missing/empty SOURCE values", {
            "platform_column": bc_platform,
            "safe_to_continue": False,
        })
    if bc_raw[source_entity_col].isna().any() or bc_raw[source_entity_col].astype(str).map(str.strip).eq("").any():
        hard_stop("BanglaClick contains missing/empty SOURCE_URL source-entity values", {
            "source_entity_column": source_entity_col,
            "safe_to_continue": False,
        })
    if bc_raw[bc_score].isna().any():
        hard_stop("BanglaClick contains missing FINAL_CLICKBAIT_LEVEL values", {
            "score_column": bc_score,
            "safe_to_continue": False,
        })

    raw_source_counts = {
        str(k): int(v)
        for k, v in bc_raw[bc_platform].value_counts(dropna=False).sort_index().items()
    }
    raw_source_values = set(bc_raw[bc_platform].astype(str).map(str.strip).unique())
    if raw_source_values != BANGLACLICK_ALLOWED_RAW_SOURCES:
        hard_stop("BanglaClick SOURCE value contract mismatch", {
            "expected_values": sorted(BANGLACLICK_ALLOWED_RAW_SOURCES),
            "observed_values": sorted(raw_source_values),
            "safe_to_continue": False,
        })

    numeric_levels = pd.to_numeric(bc_raw[bc_score], errors="coerce")
    if numeric_levels.isna().any() or ((numeric_levels % 1) != 0).any():
        hard_stop("BanglaClick FINAL_CLICKBAIT_LEVEL contains non-integer/unparseable values", {
            "observed_values": sorted(map(str, bc_raw[bc_score].dropna().unique())),
            "safe_to_continue": False,
        })
    observed_levels = {int(x) for x in numeric_levels.astype(int).unique()}
    if observed_levels != BANGLACLICK_ALLOWED_LABEL_LEVELS:
        hard_stop("BanglaClick multi-level label contract mismatch", {
            "expected_values": sorted(BANGLACLICK_ALLOWED_LABEL_LEVELS),
            "observed_values": sorted(observed_levels),
            "safe_to_continue": False,
        })

    parsed_platform = [
        parse_banglaclick_platform(s, u)
        for s, u in zip(bc_raw[bc_platform], bc_raw[BANGLACLICK_POST_URL_COLUMN])
    ]
    if any(x is None for x in parsed_platform):
        bad = bc_raw.loc[[x is None for x in parsed_platform], [bc_platform, BANGLACLICK_POST_URL_COLUMN]]
        hard_stop("BanglaClick platform/source value cannot be handled under the frozen rule", {
            "examples": bad.astype("string").head(20).to_dict(orient="records"),
            "safe_to_continue": False,
        })

    labels = bc_raw[bc_score].map(harmonize_multilevel_clickbait)
    if labels.isna().any():
        hard_stop("BanglaClick score cannot be harmonized using the frozen zero-vs-positive rule", {
            "values": sorted(map(str, bc_raw[bc_score].dropna().unique())),
            "safe_to_continue": False,
        })

    diagnostic_validation = _validate_diagnostic_columns(bc_raw)

    row_ids = [f"bc_{i:04d}" for i in range(len(bc_raw))]
    bc = pd.DataFrame({
        "row_id": row_ids,
        "title": bc_raw[bc_text].astype(str).values,
        "platform": parsed_platform,
        "label": labels.astype(int).values,
        "cluster_id": (
            "bc:"
            + pd.Series(parsed_platform, index=bc_raw.index).astype(str)
            + ":"
            + bc_raw[source_entity_col].astype(str)
        ).values,
    })
    for key, col in diag_cols.items():
        # Keep frozen diagnostic payloads in a single Arrow-safe representation.
        # Some XLSX diagnostic cells are mixed int/string objects; diagnostics are
        # descriptive only and are parsed explicitly later by diagnostic_numeric().
        bc[f"diag_{key}"] = bc_raw[col].astype("string").values

    observed_platforms = set(bc["platform"].unique())
    if observed_platforms != BANGLACLICK_REQUIRED_PLATFORMS:
        hard_stop("BanglaClick required platform-stratum contract mismatch", {
            "expected": sorted(BANGLACLICK_REQUIRED_PLATFORMS),
            "observed": sorted(observed_platforms),
            "safe_to_continue": False,
        })
    for plat, g in bc.groupby("platform", sort=True):
        if set(g["label"].unique()) != {0, 1}:
            hard_stop("BanglaClick platform stratum lacks both binary classes", {
                "platform": plat,
                "labels": sorted(map(int, g["label"].unique())),
                "safe_to_continue": False,
            })

    social_hosts = (
        bc_raw.loc[bc_raw[bc_platform].eq("Social Media"), BANGLACLICK_POST_URL_COLUMN]
        .map(_url_host)
        .value_counts(dropna=False)
    )
    platform_counts = {str(k): int(v) for k, v in bc["platform"].value_counts().sort_index().items()}
    binary_label_counts = {str(int(k)): int(v) for k, v in bc["label"].value_counts().sort_index().items()}
    multi_level_counts = {str(int(k)): int(v) for k, v in numeric_levels.astype(int).value_counts().sort_index().items()}
    clusters_by_platform = {
        str(plat): int(g["cluster_id"].nunique())
        for plat, g in bc.groupby("platform", sort=True)
    }

    schema_report = {
        "schema_version": 1,
        "validation_status": "PASS",
        "physical_workbook": {
            "path": str(Path(path)),
            "repository_relative_path": BANGLACLICK_RELATIVE_PATH,
            "source_commit": BANGLACLICK_COMMIT,
            "sha256": sha256_file(path),
            "size_bytes": int(Path(path).stat().st_size),
            "sheet_name": BANGLACLICK_EXPECTED_SHEET,
            "row_count": int(len(bc_raw)),
            "column_count": int(len(bc_raw.columns)),
        },
        "raw_headers": BANGLACLICK_EXPECTED_HEADERS,
        "normalized_headers": [ncol(x) for x in BANGLACLICK_EXPECTED_HEADERS],
        "duplicate_normalized_headers": {},
        "duplicate_normalized_header_status": "NONE",
        "role_mapping": {
            "text_column": bc_text,
            "platform_column": bc_platform,
            "score_column": bc_score,
            "diagnostic_columns": BANGLACLICK_DIAGNOSTIC_COLUMNS,
            "source_entity_column": source_entity_col,
        },
        "missingness": missingness,
        "source_values": {
            "unique_values": sorted(raw_source_values),
            "counts": raw_source_counts,
            "explicit_mapping_rule": BANGLACLICK_SCHEMA_CONTRACT["platform_value_rule"],
            "social_media_post_url_hosts": {str(k): int(v) for k, v in social_hosts.sort_index().items()},
            "unhandled_values_or_hosts": [],
        },
        "platform_values": {
            "unique_values": sorted(observed_platforms),
            "counts": platform_counts,
            "cluster_counts": clusters_by_platform,
        },
        "multi_level_label_values": {
            "unique_values": sorted(observed_levels),
            "counts": multi_level_counts,
            "explicit_binary_mapping": {"0": 0, "1": 1, "2": 1, "3": 1},
            "rejected_values": [],
        },
        "binary_label_values": {
            "unique_values": [0, 1],
            "counts": binary_label_counts,
        },
        "diagnostic_validation": diagnostic_validation,
        "selection_integrity": {
            "schema_roles_selected_from": "exact frozen physical header identity only",
            "target_label_distribution_used_for_role_selection": False,
            "model_outputs_used_for_role_selection": False,
            "performance_metrics_used_for_role_selection": False,
            "target_tuning_or_recalibration": False,
        },
    }
    write_json(OUT_ROOT / "BANGLACLICK_SCHEMA_REPORT.json", schema_report)
    print(
        "BanglaClick loader/schema validation: PASS — "
        f"rows={len(bc_raw)}, platforms={platform_counts}, levels={multi_level_counts}"
    )
    return bc_raw, bc, schema_report, probe

bc_raw, bc, BANGLACLICK_SCHEMA_REPORT, BANGLACLICK_SCHEMA_PROBE = load_banglaclick_frozen(BANGLACLICK_PATH)

bc_text = BANGLACLICK_TEXT_COLUMN
bc_platform = BANGLACLICK_PLATFORM_COLUMN
bc_score = BANGLACLICK_SCORE_COLUMN
diag_cols = {ncol(c): c for c in BANGLACLICK_DIAGNOSTIC_COLUMNS}
source_entity_col = BANGLACLICK_SOURCE_ENTITY_COLUMN
id_col = None
id_fallback = True
missing_entity_fallback_rows = 0

# BaitBuster: human labels only, title_debiased only, channel_id clusters. Read only required columns.
bb_schema=set(pq.ParquetFile(BAITBUSTER_PATH).schema.names)
for col in ["title_debiased","human_labeled","channel_id"]:
    if col not in bb_schema:hard_stop("BaitBuster required column missing",{"column":col,"columns":sorted(bb_schema)})
bb_cols=["title_debiased","human_labeled","channel_id"]+(["video_id"] if "video_id" in bb_schema else [])
bb_raw=pd.read_parquet(BAITBUSTER_PATH,columns=bb_cols)
mask=bb_raw["human_labeled"].notna();bb0=bb_raw.loc[mask].copy();labels=bb0["human_labeled"].map(harmonize_baitbuster)
if labels.isna().any():hard_stop("BaitBuster human_labeled values cannot be harmonized",{"values":sorted(map(str,bb0.loc[labels.isna(),"human_labeled"].unique()))})
if bb0["title_debiased"].map(norm_text).eq("").any():hard_stop("BaitBuster human subset has empty title_debiased")
if bb0["channel_id"].isna().any():hard_stop("BaitBuster human subset missing channel_id")
rowid=bb0["video_id"].astype(str) if "video_id" in bb0.columns else pd.Series([f"bb_{i}" for i in bb0.index],index=bb0.index)
bb=pd.DataFrame({"row_id":rowid.values,"title":bb0["title_debiased"].astype(str).values,"platform":"youtube",
                 "label":labels.astype(int).values,"cluster_id":("bb:"+bb0["channel_id"].astype(str)).values})
if bb["row_id"].duplicated().any():hard_stop("BaitBuster human subset row IDs are not unique")
if set(bb["label"].unique())!={0,1}:hard_stop("BaitBuster human subset lacks both binary classes",{"labels":sorted(map(int,bb["label"].unique())),"safe_to_continue":False})

# Source TEST comes only from immutable Stage-03 clean artifact.
test_rec=MODEL_MAN["clean_split_artifacts"]["test"];test_path=S3/test_rec["path"]
if sha256_file(test_path)!=test_rec["sha256"]:hard_stop("Clean source TEST hash mismatch")
src0=pd.read_parquet(test_path).sort_values("source_row_id",kind="stable").reset_index(drop=True)
src=pd.DataFrame({"row_id":src0["source_row_id"].astype(str),"title":src0["title"].astype(str),"platform":"source_news",
                  "label":src0["label"].astype(int),"cluster_id":"source_row:"+src0["source_row_id"].astype(str)})

adapter_report={
    "schema_version":1,
    "banglaclick":{"physical_rows":int(len(bc_raw)),"analysis_rows":int(len(bc)),"sheet_name":BANGLACLICK_EXPECTED_SHEET,"text_column":str(bc_text),"platform_column":str(bc_platform),"score_column":str(bc_score),
                   "diagnostic_columns":{k:str(v) for k,v in diag_cols.items()},"source_entity_column":str(source_entity_col),
                   "row_id_column":None,"row_id_fallback_due_missing_or_duplicates":True,"source_entity_missing_row_fallback_count":0,
                   "workbook_sha256":BANGLACLICK_SHA256,"schema_report_sha256":sha256_file(OUT_ROOT/"BANGLACLICK_SCHEMA_REPORT.json"),
                   "platform_counts":{str(k):int(v) for k,v in bc.platform.value_counts().sort_index().items()},"positive_prevalence":float(bc.label.mean())},
    "baitbuster":{"physical_rows":int(pq.ParquetFile(BAITBUSTER_PATH).metadata.num_rows),"human_labeled_rows":int(len(bb)),"excluded_nonhuman_rows":int(pq.ParquetFile(BAITBUSTER_PATH).metadata.num_rows-len(bb)),
                  "text_column":"title_debiased","label_column":"human_labeled","cluster_column":"channel_id","positive_prevalence":float(bb.label.mean())},
    "source_test":{"rows":int(len(src)),"positive_prevalence":float(src.label.mean()),"artifact_sha256":test_rec["sha256"]},
    "harmonization":{"banglaclick":"dataset-defined zero/no-clickbait => 0; explicit positive levels => 1","baitbuster":"non-null human_labeled only; binary semantic mapping"},
}
write_json(OUT_ROOT/"EXTERNAL_DATA_ADAPTER_REPORT.json",adapter_report)
print("Dataset sizes — source TEST",len(src),"BanglaClick",len(bc),"BaitBuster human",len(bb))


BanglaClick exact physical schema: PASS
{
  "schema_version": 3,
  "path": "/kaggle/input/datasets/smalakarishere/swarabyanjan-frozen-dataset-v1/EXTERNAL-DATASET-BYTES-v1/EXTERNAL-DATASET-BYTES-v1/BanglaClick/repository_snapshot/Data/Final_4961_Dataset.xlsx",
  "filename": "Final_4961_Dataset.xlsx",
  "repository_relative_path": "BanglaClick/repository_snapshot/Data/Final_4961_Dataset.xlsx",
  "source_commit": "2ba28763d8f80eaba7a649877eae1fbbcb78a0e7",
  "workbook_size": 1277552,
  "workbook_sha256": "2202f89445fa12f1880854edb50f10d37293bd006758fe1901c69213f058e060",
  "workbook_sheet_names": [
    "Sheet1"
  ],
  "sheet_name": "Sheet1",
  "excel_max_row_including_header": 4962,
  "column_count": 14,
  "raw_headers": [
    "SOURCE",
    "SOURCE_URL",
    "POST_URL",
    "TEXT",
    "CONTENT",
    "CONTENT_URL",
    "MEDIA_TYPE",
    "MEDIA_LINK",
    "FAST_FAIL_COUNT",
    "FAST_LEVEL",
    "STRONG_TRIGGER_COUNT",
    "WEAK_TRIGGER_COUNT",
    "TRIGGER_LEVEL",
    "FINAL_CLICKBAIT_LEV

## Method
The verified Stage 03 model panel and Stage 04 source-DEV temperatures are applied without target fitting. H3 uses the prespecified cue schema; H3-P neutralizes only frozen cue spans/punctuation as model-sensitivity corroboration.

## Evaluation
Locked source TEST, platform-stratified BanglaClick, and BaitBuster human-labelled replication are evaluated with the prespecified discrimination, proper-loss, calibration, cue-transport, and perturbation analyses.

### Locked Inference and Reliability Evaluation
Models and temperatures are loaded only from verified upstream artifacts. Inference may reduce physical batch size after CUDA OOM; this does not change model parameters, data, seeds, thresholds, or scientific estimands.


In [7]:
# ---------- Frozen model inference + source-DEV temperatures ----------
import torch
from torch.utils.data import Dataset,DataLoader
from transformers import AutoTokenizer,AutoModelForSequenceClassification

if not torch.cuda.is_available():
    hard_stop("CUDA GPU is unavailable for Stage-05 transformer inference and perturbation",
              {"safe_to_continue":False,"action":"Enable a Kaggle GPU accelerator and rerun Stage 05 from the top."})
print("GPU:",torch.cuda.get_device_name(0))

class InferDS(Dataset):
    def __init__(self,texts,tok,max_length):self.enc=tok(list(texts),truncation=True,padding=False,max_length=max_length)
    def __len__(self):return len(next(iter(self.enc.values())))
    def __getitem__(self,i):return {k:torch.tensor(v[i],dtype=torch.long) for k,v in self.enc.items()}
def infer_transformer(path,texts,max_length,batch=64):
    tok=AutoTokenizer.from_pretrained(path,local_files_only=True);model=AutoModelForSequenceClassification.from_pretrained(path,local_files_only=True)
    ds=InferDS(texts,tok,max_length);dev=torch.device("cuda");model.to(dev);model.eval();last_error=None
    tried=[]
    for bs in [batch,32,16,8]:
        bs=min(bs,batch)
        if bs in tried or bs<=0:continue
        tried.append(bs)
        try:
            loader=DataLoader(ds,batch_size=bs,shuffle=False,collate_fn=lambda b:tok.pad(b,padding=True,return_tensors="pt"));z=[]
            with torch.no_grad():
                for ba in loader:
                    ba={k:v.to(dev) for k,v in ba.items()};lo=model(**ba).logits.detach().cpu().numpy();z.extend((lo[:,1]-lo[:,0]).tolist())
            del model;torch.cuda.empty_cache();return np.asarray(z,float)
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():raise
            last_error=e;torch.cuda.empty_cache()
    hard_stop("CUDA out of memory during transformer inference",{"attempted_batch_sizes":tried,"safe_to_continue":False,"error":str(last_error)})
def model_rec(m,s):
    r=[x for x in MODEL_MAN["models"] if x["model"]==m and int(x["seed"])==int(s)]
    if len(r)!=1:hard_stop("Model artifact record ambiguity",{"model":m,"seed":s})
    return r[0]
def verify_model_rec(rec):
    for f in rec["files"]:
        p=S3/f["path"]
        if not p.is_file() or sha256_file(p)!=f["sha256"]:hard_stop("Model file hash mismatch",{"path":str(p)})
def infer_dataset(df,dataset_name):
    base=df.reset_index(drop=True).copy();features=cue_features(base["title"].tolist())
    for c in features.columns:base[c]=features[c].values
    base["normalized_title_sha256"]=[hashlib.sha256(norm_text(x).encode("utf-8")).hexdigest() for x in base["title"]]
    allrows=[]
    for m in MODEL_KEYS:
        for seed in SEEDS:
            rec=model_rec(m,seed);verify_model_rec(rec);p=S3/rec["artifact_path"]
            if m=="ebm":z=np.asarray(joblib.load(p).decision_function(features.astype(float)),float)
            elif m=="tfidf_lr":z=np.asarray(joblib.load(p).decision_function(base["title"]),float)
            else:z=infer_transformer(p,base["title"].tolist(),int(CONFIG["models"][m]["base_config"]["max_length"]))
            T=float(TEMPS[m][str(seed)]);q=base.copy();q["dataset"]=dataset_name;q["model"]=m;q["seed"]=seed;q["logit"]=z;q["p_uncalibrated"]=expit(z);q["temperature"]=T;q["p_calibrated"]=expit(z/T)
            allrows.append(q)
    return pd.concat(allrows,ignore_index=True)

SRC_P=infer_dataset(src,"source_test")
BC_P=infer_dataset(bc,"banglaclick")
BB_P=infer_dataset(bb,"baitbuster_human")
SRC_P.to_parquet(OUT_ROOT/"LOCKED_SOURCE_TEST_PREDICTIONS.parquet",index=False)
BC_P.to_parquet(OUT_ROOT/"BANGLACLICK_PLATFORM_PREDICTIONS.parquet",index=False)
BB_P.to_parquet(OUT_ROOT/"BAITBUSTER_HUMAN_PREDICTIONS.parquet",index=False)
print("Locked inference artifacts written")


GPU: Tesla T4


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Locked inference artifacts written


In [8]:
# ---------- Metric definitions and H1/H2 ----------
print("[H1/H2 metrics] START — deterministic point metrics and sensitivity metrics", flush=True)
def balanced_brier(y,p,w=None):
    y=np.asarray(y,int);p=np.asarray(p,float);w=None if w is None else np.asarray(w,float);vals=[]
    for c in [0,1]:
        m=y==c
        if not m.any():return None
        vals.append(float(np.average((p[m]-y[m])**2,weights=None if w is None else w[m])))
    return 0.5*sum(vals)
def calib_is(y,p,w=None):
    p=np.clip(np.asarray(p,float),1e-7,1-1e-7);X=sm.add_constant(logit(p),has_constant="add")
    try:
        fit=sm.GLM(np.asarray(y,int),X,family=sm.families.Binomial(),freq_weights=w).fit()
        return float(fit.params[0]),float(fit.params[1]),None
    except (ValueError, np.linalg.LinAlgError, RuntimeError) as exc:
        return None,None,f"{type(exc).__name__}: {exc}"
def metric_bundle(y,p,w=None):
    y=np.asarray(y,int);p=np.clip(np.asarray(p,float),1e-7,1-1e-7);w=None if w is None else np.asarray(w,float)
    if len(np.unique(y))<2:
        return {"estimable":False,"reason":"single_class","auroc":None,"auprc":None,"nll":None,"brier":None,"balanced_brier":None,"calibration_intercept":None,"calibration_slope":None,"calibration_fit_issue":"single_class","f1_at_0_50":None}
    if w is not None and (not np.isfinite(w).all() or w.sum()<=0 or any(w[y==c].sum()<=0 for c in [0,1])):
        return {"estimable":False,"reason":"nonpositive_or_class-degenerate_weights","auroc":None,"auprc":None,"nll":None,"brier":None,"balanced_brier":None,"calibration_intercept":None,"calibration_slope":None,"calibration_fit_issue":"nonpositive_or_class-degenerate_weights","f1_at_0_50":None}
    ci,cs,fit_issue=calib_is(y,p,w)
    nll=float(np.average(-(y*np.log(p)+(1-y)*np.log(1-p)),weights=w));brier=float(np.average((p-y)**2,weights=w))
    return {"estimable":True,"auroc":float(roc_auc_score(y,p,sample_weight=w)),"auprc":float(average_precision_score(y,p,sample_weight=w)),
            "nll":nll,"brier":brier,"balanced_brier":balanced_brier(y,p,w),"calibration_intercept":ci,"calibration_slope":cs,
            "calibration_fit_issue":fit_issue,"f1_at_0_50":float(f1_score(y,p>=0.5,sample_weight=w))}
def source_prior_weights(y,target_pi):
    y=np.asarray(y,int); pi=float(y.mean());
    if pi in {0,1}:return np.ones(len(y))
    return np.where(y==1,target_pi/pi,(1-target_pi)/(1-pi))
def structural_weights(source_features,target_features):
    # Unsupervised composition standardization: source quartiles of token count and source median split of code-mixing.
    q=np.unique(np.quantile(source_features["token_count"],[0,.25,.5,.75,1]))
    if len(q)<2:q=np.array([-np.inf,np.inf])
    else:q[0],q[-1]=-np.inf,np.inf
    smed=float(source_features["latin_code_mixing_proportion"].median())
    def strata(df):
        tb=np.digitize(df["token_count"].to_numpy(),q[1:-1],right=True); cb=(df["latin_code_mixing_proportion"].to_numpy()>smed).astype(int);return tb*2+cb
    ss=strata(source_features);ts=strata(target_features);sp=Counter(ss);tp=Counter(ts);N=len(ss);M=len(ts)
    return np.array([(sp.get(x,0)/N)/(tp.get(x,1)/M) if tp.get(x,0)>0 and sp.get(x,0)>0 else 0.0 for x in ts],float)

# Domain frames: each seed/model appears repeatedly but unique item rows are stable.
domain_frames={"source_test":SRC_P}
for plat in ["news","facebook","youtube","twitter"]:domain_frames[f"banglaclick:{plat}"]=BC_P[BC_P["platform"]==plat].copy()
domain_frames["baitbuster:youtube_human"]=BB_P
source_item=SRC_P[(SRC_P.model=="ebm")&(SRC_P.seed==SEEDS[0])].drop_duplicates("row_id")
source_pi=float(source_item.label.mean())

metric_rows=[]
for dname,df in domain_frames.items():
    for (m,s),g in df.groupby(["model","seed"],sort=True):
        g=g.sort_values("row_id",kind="stable");y=g.label.to_numpy();pu=g.p_uncalibrated.to_numpy();pc=g.p_calibrated.to_numpy()
        rec={"domain":dname,"model":m,"seed":int(s),"n":len(g),"prevalence":float(y.mean()),
             "uncalibrated":metric_bundle(y,pu),"calibrated":metric_bundle(y,pc)}
        if dname!="source_test":
            wp=source_prior_weights(y,source_pi);rec["source_prior_standardized"]={"uncalibrated":metric_bundle(y,pu,wp),"calibrated":metric_bundle(y,pc,wp)}
            tgt=g.drop_duplicates("row_id")[["token_count","latin_code_mixing_proportion"]]
            srcf=source_item[["token_count","latin_code_mixing_proportion"]];ws=structural_weights(srcf,tgt)
            # expand item weights in this single model/seed group one-to-one
            rec["structural_composition_standardized"]={"calibrated":metric_bundle(y,pc,ws)}
        metric_rows.append(rec)

# H1 support screen per model/seed/target using source calibrated comparison.
source_lookup={(r["model"],r["seed"]):r for r in metric_rows if r["domain"]=="source_test"}
h1_screen=[];h2_screen=[]
for r in metric_rows:
    if r["domain"]=="source_test":continue
    srcm=source_lookup[(r["model"],r["seed"])]
    auc_drop=srcm["calibrated"]["auroc"]-r["calibrated"]["auroc"]
    nll_rel=(r["calibrated"]["nll"]-srcm["calibrated"]["nll"])/srcm["calibrated"]["nll"]
    bbs_rel=(r["calibrated"]["balanced_brier"]-srcm["calibrated"]["balanced_brier"])/srcm["calibrated"]["balanced_brier"]
    h1_screen.append({"domain":r["domain"],"model":r["model"],"seed":r["seed"],"target_auroc":r["calibrated"]["auroc"],"auroc_drop":auc_drop,
                      "nll_relative_change":nll_rel,"balanced_brier_relative_change":bbs_rel,
                      "descriptive_h1_condition":bool(r["calibrated"]["auroc"]>=.70 and auc_drop<=.05 and (nll_rel>=.10 or bbs_rel>=.10))})
    gs=srcm["uncalibrated"]["nll"]-srcm["calibrated"]["nll"];gt=r["uncalibrated"]["nll"]-r["calibrated"]["nll"]
    h2_screen.append({"domain":r["domain"],"model":r["model"],"seed":r["seed"],"source_nll_gain":gs,"target_nll_gain":gt,"target_minus_source_gain":gt-gs})
write_json(OUT_ROOT/"H1_RESULTS.json",{"schema_version":1,"primary_metrics":"calibrated deployment probabilities","metric_rows":metric_rows,"descriptive_support_screen":h1_screen,"bootstrap_inference":"STATISTICAL_SUMMARY.json"})
write_json(OUT_ROOT/"H2_RESULTS.json",{"schema_version":1,"definition":"NLL gain = NLL_uncalibrated - NLL_temperature_scaled","rows":h2_screen,"bootstrap_inference":"STATISTICAL_SUMMARY.json"})
print(f"[H1/H2 metrics] COMPLETE — metric_rows={len(metric_rows)}", flush=True)


[H1/H2 metrics] START — deterministic point metrics and sensitivity metrics
[H1/H2 metrics] COMPLETE — metric_rows=120


In [9]:
# ---------- Efficient cluster bootstrap utilities + H1/H2 inference ----------
import time

BOOTSTRAP_PROGRESS_EVERY = 1000
BOOTSTRAP_PHASE_LIMIT_SECONDS = 7200  # per major bootstrap phase; diagnostic hard-stop, not a notebook-wide timeout
CHECKPOINT_DIR = OUT_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def bootstrap_ci(values):
    a=np.asarray(values,float);a=a[np.isfinite(a)]
    return [float(np.percentile(a,2.5)),float(np.percentile(a,97.5))] if len(a) else [None,None]

def bootstrap_p_two_sided(values):
    a=np.asarray(values,float);a=a[np.isfinite(a)]
    if not len(a):return None
    return float(min(1.0,2*min(np.mean(a<=0),np.mean(a>=0))))

def simes(pvals):
    p=np.sort(np.asarray([p for p in pvals if p is not None and np.isfinite(p)],float));m=len(p)
    return float(min(1.0,np.min(p*m/np.arange(1,m+1)))) if m else None

class ClusterBootstrapPlan:
    """
    Freeze cluster membership once. A bootstrap draw samples the same number of
    clusters with replacement as the original implementation, but returns
    per-row multiplicities instead of rebuilding cluster->row mappings.
    """
    def __init__(self, cluster_ids, name):
        clusters=np.asarray(cluster_ids).astype(str)
        if len(clusters)==0:
            hard_stop("Cannot build cluster bootstrap plan for an empty frame",{"name":name})
        uniq=np.array(sorted(set(clusters)))
        code={u:i for i,u in enumerate(uniq)}
        self.row_cluster=np.asarray([code[u] for u in clusters],dtype=np.int32)
        self.cluster_labels=uniq
        self.n_clusters=int(len(uniq))
        self.n_rows=int(len(clusters))
        self.name=str(name)

    def draw_row_counts(self, rng):
        # rng.choice(n_clusters, ...) consumes the same categorical bootstrap law
        # as choosing from the sorted string labels, without string comparisons.
        sampled_codes=rng.choice(self.n_clusters,size=self.n_clusters,replace=True)
        cluster_multiplicity=np.bincount(sampled_codes,minlength=self.n_clusters)
        return cluster_multiplicity[self.row_cluster]

def _phase_guard(phase, phase_start, completed=None, total=None, context=None):
    elapsed=time.monotonic()-phase_start
    if elapsed > BOOTSTRAP_PHASE_LIMIT_SECONDS:
        hard_stop(
            f"{phase} exceeded the phase runtime guard",
            {
                "elapsed_seconds":float(elapsed),
                "phase_limit_seconds":BOOTSTRAP_PHASE_LIMIT_SECONDS,
                "completed":completed,
                "total":total,
                "context":context or {},
                "safe_to_continue":False,
                "action":"Inspect the latest checkpoint/progress record before rerunning this statistical phase.",
            },
        )
    return elapsed

def _progress(phase, done, total, phase_start, detail=""):
    elapsed=_phase_guard(phase,phase_start,done,total,{"detail":detail})
    rate=done/elapsed if elapsed>0 else None
    eta=(total-done)/rate if rate and rate>0 else None
    print(
        f"[{phase}] {done}/{total} ({100.0*done/total:.1f}%) "
        f"elapsed={elapsed:.1f}s eta={eta:.1f}s {detail}".rstrip(),
        flush=True,
    )

def _auc_setup(y,p):
    y=np.asarray(y,int);p=np.asarray(p,float)
    order=np.argsort(p,kind="mergesort")
    ps=p[order]
    starts=np.r_[0,np.flatnonzero(ps[1:]!=ps[:-1])+1]
    return order,starts

def _auc_from_row_counts(y,row_counts,setup):
    """
    Exact weighted Mann-Whitney AUROC with 0.5 tie credit.
    Row multiplicities are mathematically identical to materializing the
    resampled rows, but avoid repeated DataFrame/array expansion.
    """
    y=np.asarray(y,int);w=np.asarray(row_counts,float)
    order,starts=setup
    ys=y[order];ws=w[order]
    pos=np.add.reduceat(ws*ys,starts)
    neg=np.add.reduceat(ws*(1-ys),starts)
    P=float(pos.sum());N=float(neg.sum())
    if P<=0 or N<=0:return np.nan
    neg_before=np.cumsum(neg)-neg
    return float(np.sum(pos*(neg_before+0.5*neg))/(P*N))

def _weighted_mean(v,w):
    v=np.asarray(v,float);w=np.asarray(w,float);den=float(w.sum())
    return float(np.dot(v,w)/den) if den>0 else np.nan

def _core_boot_metrics_from_counts(y,p,row_counts,auc_setup,loss=None,sqerr=None):
    y=np.asarray(y,int);p=np.asarray(p,float);w=np.asarray(row_counts,float)
    w0=float(w[y==0].sum());w1=float(w[y==1].sum())
    if w0<=0 or w1<=0:return None
    pp=np.clip(p,1e-7,1-1e-7)
    if loss is None:loss=-(y*np.log(pp)+(1-y)*np.log(1-pp))
    if sqerr is None:sqerr=(pp-y)**2
    auc=_auc_from_row_counts(y,w,auc_setup)
    nll=_weighted_mean(loss,w)
    bb=0.5*(_weighted_mean(sqerr[y==0],w[y==0])+_weighted_mean(sqerr[y==1],w[y==1]))
    return auc,nll,bb

def _nll_from_counts(loss,row_counts):
    return _weighted_mean(loss,row_counts)

# Build one immutable cluster plan per domain. Cluster membership is invariant
# across model and seed predictions, so rebuilding it 5,000 times is unnecessary.
domain_boot_plans={}
domain_boot_row_ids={}
for dname,df in domain_frames.items():
    base=(
        df[(df.model==MODEL_KEYS[0])&(df.seed==SEEDS[0])]
        .sort_values("row_id",kind="stable")
        .reset_index(drop=True)
    )
    if base.empty:
        hard_stop("Bootstrap domain has no canonical rows",{"domain":dname})
    domain_boot_plans[dname]=ClusterBootstrapPlan(base.cluster_id.to_numpy(),dname)
    domain_boot_row_ids[dname]=base.row_id.astype(str).tolist()

rng=np.random.default_rng(BOOTSTRAP_SEED)
stat_rows=[]
targets=[d for d in domain_frames if d!="source_test"]
total_settings=len(targets)*len(MODEL_KEYS)*len(SEEDS)
setting_done=0
phase_start=time.monotonic()
print(
    f"[H1/H2 bootstrap] START — settings={total_settings}, "
    f"replicates_per_setting={BOOTSTRAP_REPS}, cluster plans precomputed once",
    flush=True,
)

for target_name in targets:
    target_df=domain_frames[target_name]
    for m in MODEL_KEYS:
        for seed in SEEDS:
            setting_done+=1
            detail=f"setting={setting_done}/{total_settings} target={target_name} model={m} seed={seed}"
            print(f"[H1/H2 bootstrap] SETTING START — {detail}",flush=True)

            s=(
                SRC_P[(SRC_P.model==m)&(SRC_P.seed==seed)]
                .sort_values("row_id",kind="stable")
                .reset_index(drop=True)
            )
            t=(
                target_df[(target_df.model==m)&(target_df.seed==seed)]
                .sort_values("row_id",kind="stable")
                .reset_index(drop=True)
            )
            if s.row_id.astype(str).tolist()!=domain_boot_row_ids["source_test"]:
                hard_stop("Source prediction row order changed relative to bootstrap plan",{"model":m,"seed":seed})
            if t.row_id.astype(str).tolist()!=domain_boot_row_ids[target_name]:
                hard_stop("Target prediction row order changed relative to bootstrap plan",{"domain":target_name,"model":m,"seed":seed})

            sy=s.label.to_numpy(int);ty=t.label.to_numpy(int)
            sp=np.clip(s.p_calibrated.to_numpy(float),1e-7,1-1e-7)
            tp=np.clip(t.p_calibrated.to_numpy(float),1e-7,1-1e-7)
            su=np.clip(s.p_uncalibrated.to_numpy(float),1e-7,1-1e-7)
            tu=np.clip(t.p_uncalibrated.to_numpy(float),1e-7,1-1e-7)

            s_loss=-(sy*np.log(sp)+(1-sy)*np.log(1-sp))
            t_loss=-(ty*np.log(tp)+(1-ty)*np.log(1-tp))
            su_loss=-(sy*np.log(su)+(1-sy)*np.log(1-su))
            tu_loss=-(ty*np.log(tu)+(1-ty)*np.log(1-tu))
            s_sq=(sp-sy)**2;t_sq=(tp-ty)**2
            s_auc_setup=_auc_setup(sy,sp);t_auc_setup=_auc_setup(ty,tp)

            spl=domain_boot_plans["source_test"];tpl=domain_boot_plans[target_name]
            aucd=[];nlld=[];bbd=[];source_gain=[];target_gain=[];gain_diff=[]

            for rep in range(1,BOOTSTRAP_REPS+1):
                sw=spl.draw_row_counts(rng);tw=tpl.draw_row_counts(rng)
                smet=_core_boot_metrics_from_counts(sy,sp,sw,s_auc_setup,s_loss,s_sq)
                tmet=_core_boot_metrics_from_counts(ty,tp,tw,t_auc_setup,t_loss,t_sq)
                if smet is not None and tmet is not None:
                    sauc,snll,sbb=smet;tauc,tnll,tbb=tmet
                    aucd.append(sauc-tauc)
                    nlld.append((tnll-snll)/snll)
                    bbd.append((tbb-sbb)/sbb)
                    gs=_nll_from_counts(su_loss,sw)-snll
                    gt=_nll_from_counts(tu_loss,tw)-tnll
                    source_gain.append(gs);target_gain.append(gt);gain_diff.append(gt-gs)

                if rep%100==0:
                    _phase_guard("H1/H2 bootstrap",phase_start,rep,BOOTSTRAP_REPS,{"setting":detail})
                if rep%BOOTSTRAP_PROGRESS_EVERY==0 or rep==BOOTSTRAP_REPS:
                    _progress("H1/H2 bootstrap",rep,BOOTSTRAP_REPS,phase_start,detail)

            stat_rows.append({
                "domain":target_name,"model":m,"seed":seed,
                "bootstrap_attempts":BOOTSTRAP_REPS,
                "bootstrap_valid_two_class":len(gain_diff),
                "auroc_drop_ci95":bootstrap_ci(aucd),
                "nll_relative_change_ci95":bootstrap_ci(nlld),
                "balanced_brier_relative_change_ci95":bootstrap_ci(bbd),
                "h2_source_gain_ci95":bootstrap_ci(source_gain),
                "h2_target_gain_ci95":bootstrap_ci(target_gain),
                "h2_target_minus_source_gain_ci95":bootstrap_ci(gain_diff),
                "h2_target_minus_source_gain_p":bootstrap_p_two_sided(gain_diff),
            })
            write_json(
                CHECKPOINT_DIR/"H1_H2_BOOTSTRAP_CHECKPOINT.json",
                {
                    "schema_version":2,
                    "phase":"H1/H2 bootstrap",
                    "completed_settings":setting_done,
                    "total_settings":total_settings,
                    "replicates_per_setting":BOOTSTRAP_REPS,
                    "rng_state":rng.bit_generator.state,
                    "rows":stat_rows,
                    "last_completed":{"domain":target_name,"model":m,"seed":int(seed)},
                },
            )
            print(f"[H1/H2 bootstrap] SETTING COMPLETE — {detail} valid={len(gain_diff)}",flush=True)

write_json(
    OUT_ROOT/"STATISTICAL_SUMMARY.json",
    {
        "schema_version":1,
        "bootstrap_replicates":BOOTSTRAP_REPS,
        "ci":"95% percentile",
        "cluster_rule":"strongest available true source entity; source falls back to row",
        "h1_h2_rows":stat_rows,
    },
)
print(
    f"[H1/H2 bootstrap] COMPLETE — settings={len(stat_rows)} "
    f"elapsed={time.monotonic()-phase_start:.1f}s",
    flush=True,
)


[H1/H2 bootstrap] START — settings=100, replicates_per_setting=5000, cluster plans precomputed once
[H1/H2 bootstrap] SETTING START — setting=1/100 target=banglaclick:news model=ebm seed=42
[H1/H2 bootstrap] 1000/5000 (20.0%) elapsed=0.5s eta=1.9s setting=1/100 target=banglaclick:news model=ebm seed=42
[H1/H2 bootstrap] 2000/5000 (40.0%) elapsed=0.9s eta=1.4s setting=1/100 target=banglaclick:news model=ebm seed=42
[H1/H2 bootstrap] 3000/5000 (60.0%) elapsed=1.4s eta=0.9s setting=1/100 target=banglaclick:news model=ebm seed=42
[H1/H2 bootstrap] 4000/5000 (80.0%) elapsed=1.9s eta=0.5s setting=1/100 target=banglaclick:news model=ebm seed=42
[H1/H2 bootstrap] 5000/5000 (100.0%) elapsed=2.3s eta=0.0s setting=1/100 target=banglaclick:news model=ebm seed=42
[H1/H2 bootstrap] SETTING COMPLETE — setting=1/100 target=banglaclick:news model=ebm seed=42 valid=5000
[H1/H2 bootstrap] SETTING START — setting=2/100 target=banglaclick:news model=ebm seed=137
[H1/H2 bootstrap] 1000/5000 (20.0%) elapsed=

## H3 — Cue Transportability
This analysis estimates prespecified cue–outcome and cue-conditioned proper-loss transport under observed distribution shift. Results are association/transport evidence, not causal platform effects.


In [10]:
# ---------- H3 observational cue transport: optimized exact cluster bootstrap ----------
def item_frame(pred):
    return pred[(pred.model=="ebm")&(pred.seed==SEEDS[0])].drop_duplicates("row_id").sort_values("row_id",kind="stable").reset_index(drop=True)

def ebm_seed_frames(pred):
    frames={}
    canonical_ids=None
    for es in SEEDS:
        g=pred[(pred.model=="ebm")&(pred.seed==es)].sort_values("row_id",kind="stable").reset_index(drop=True)
        ids=g.row_id.astype(str).tolist()
        if canonical_ids is None:canonical_ids=ids
        elif ids!=canonical_ids:hard_stop("EBM seed predictions are not row-aligned for H3 reliability transport",{"seed":es})
        frames[es]=g
    return frames

def _group_gap_1d(values,cue_present,row_counts):
    values=np.asarray(values,float);c=np.asarray(cue_present,bool);w=np.asarray(row_counts,float)
    wp=w[c];wa=w[~c]
    dp=float(wp.sum());da=float(wa.sum())
    if dp<=0 or da<=0:return np.nan
    return float(np.dot(values[c],wp)/dp - np.dot(values[~c],wa)/da)

def _group_gap_by_seed(loss_by_seed,cue_present,row_counts):
    losses=np.asarray(loss_by_seed,float);c=np.asarray(cue_present,bool);w=np.asarray(row_counts,float)
    wp=w[c];wa=w[~c]
    dp=float(wp.sum());da=float(wa.sum())
    if dp<=0 or da<=0:return np.full(losses.shape[0],np.nan,float)
    return (losses[:,c]@wp)/dp - (losses[:,~c]@wa)/da

def _h3_bundle(item,seed_frames,name):
    ids=item.row_id.astype(str).tolist()
    y=item.label.to_numpy(int)
    cues={cue:(item[cue].to_numpy()>0) for cue in CUE_FAMILIES}
    losses=[]
    for es in SEEDS:
        g=seed_frames[es]
        if g.row_id.astype(str).tolist()!=ids:
            hard_stop("H3 seed frame is not row-aligned with item frame",{"domain":name,"seed":es})
        p=np.clip(g.p_calibrated.to_numpy(float),1e-7,1-1e-7)
        yy=g.label.to_numpy(int)
        if not np.array_equal(yy,y):
            hard_stop("H3 label mismatch across EBM seeds",{"domain":name,"seed":es})
        losses.append(-(yy*np.log(p)+(1-yy)*np.log(1-p)))
    return {
        "name":name,
        "y":y,
        "cues":cues,
        "loss_by_seed":np.vstack(losses),
        "plan":ClusterBootstrapPlan(item.cluster_id.to_numpy(),name),
        "ones":np.ones(len(item),dtype=np.int16),
    }

src_item=item_frame(SRC_P)
bc_items={p:item_frame(BC_P[BC_P.platform==p]) for p in ["news","facebook","youtube","twitter"]}
bb_item=item_frame(BB_P)
src_ebm=ebm_seed_frames(SRC_P)
bc_ebm={p:ebm_seed_frames(BC_P[BC_P.platform==p]) for p in ["news","facebook","youtube","twitter"]}
bb_ebm=ebm_seed_frames(BB_P)

src_h3=_h3_bundle(src_item,src_ebm,"source_test")
bc_h3={p:_h3_bundle(bc_items[p],bc_ebm[p],f"banglaclick:{p}") for p in ["news","facebook","youtube","twitter"]}
bb_h3=_h3_bundle(bb_item,bb_ebm,"baitbuster:youtube_human")

cue_transport=[]
rng_h3=np.random.default_rng(BOOTSTRAP_SEED+1)
h3_start=time.monotonic()
total_cp=len(CUE_FAMILIES)*len(bc_h3)
cp_done=0
print(
    f"[H3 cue/platform bootstrap] START — contrasts={total_cp}, replicates_per_contrast={BOOTSTRAP_REPS}",
    flush=True,
)

for cue in CUE_FAMILIES:
    for plat in ["news","facebook","youtube","twitter"]:
        cp_done+=1
        tgt=bc_h3[plat]
        src_rd=_group_gap_1d(src_h3["y"],src_h3["cues"][cue],src_h3["ones"])
        tgt_rd=_group_gap_1d(tgt["y"],tgt["cues"][cue],tgt["ones"])
        src_nll_seed=_group_gap_by_seed(src_h3["loss_by_seed"],src_h3["cues"][cue],src_h3["ones"])
        tgt_nll_seed=_group_gap_by_seed(tgt["loss_by_seed"],tgt["cues"][cue],tgt["ones"])
        by_seed=tgt_nll_seed-src_nll_seed
        point_rd=tgt_rd-src_rd
        point_nll=float(np.nanmean(by_seed))

        rd_boot=[];nll_boot=[]
        detail=f"contrast={cp_done}/{total_cp} cue={cue} platform={plat}"
        print(f"[H3 cue/platform bootstrap] CONTRAST START — {detail}",flush=True)
        for rep in range(1,BOOTSTRAP_REPS+1):
            sw=src_h3["plan"].draw_row_counts(rng_h3)
            tw=tgt["plan"].draw_row_counts(rng_h3)
            rd_boot.append(
                _group_gap_1d(tgt["y"],tgt["cues"][cue],tw)
                - _group_gap_1d(src_h3["y"],src_h3["cues"][cue],sw)
            )
            sd=_group_gap_by_seed(src_h3["loss_by_seed"],src_h3["cues"][cue],sw)
            td=_group_gap_by_seed(tgt["loss_by_seed"],tgt["cues"][cue],tw)
            nll_boot.append(float(np.nanmean(td-sd)))

            if rep%100==0:
                _phase_guard("H3 cue/platform bootstrap",h3_start,rep,BOOTSTRAP_REPS,{"contrast":detail})
            if rep%BOOTSTRAP_PROGRESS_EVERY==0 or rep==BOOTSTRAP_REPS:
                _progress("H3 cue/platform bootstrap",rep,BOOTSTRAP_REPS,h3_start,detail)

        p_rd=bootstrap_p_two_sided(rd_boot);p_nll=bootstrap_p_two_sided(nll_boot)
        cue_transport.append({
            "cue":cue,"target":f"banglaclick:{plat}",
            "source_rd":src_rd,"target_rd":tgt_rd,
            "delta_rd":point_rd,"delta_rd_ci95":bootstrap_ci(rd_boot),"delta_rd_p":p_rd,
            "ebm_delta_nll_gap_mean_5seeds":point_nll,
            "ebm_delta_nll_gap_ci95":bootstrap_ci(nll_boot),
            "ebm_delta_nll_gap_p":p_nll,
            "ebm_delta_nll_gap_by_seed":[float(x) if np.isfinite(x) else None for x in by_seed],
        })
        write_json(
            CHECKPOINT_DIR/"H3_CUE_PLATFORM_CHECKPOINT.json",
            {
                "schema_version":2,
                "completed_contrasts":cp_done,
                "total_contrasts":total_cp,
                "replicates_per_contrast":BOOTSTRAP_REPS,
                "rng_state":rng_h3.bit_generator.state,
                "rows":cue_transport,
                "last_completed":{"cue":cue,"platform":plat},
            },
        )
        print(f"[H3 cue/platform bootstrap] CONTRAST COMPLETE — {detail}",flush=True)

family_praw_outcome={}
family_praw_reliability={}
for cue in CUE_FAMILIES:
    rows=[r for r in cue_transport if r["cue"]==cue]
    family_praw_outcome[cue]=simes([r["delta_rd_p"] for r in rows])
    family_praw_reliability[cue]=simes([r["ebm_delta_nll_gap_p"] for r in rows])

def bh_family(pmap):
    valid=[c for c in CUE_FAMILIES if pmap[c] is not None]
    if not valid:return {}
    rej,qvals,_,_=multipletests([pmap[c] for c in valid],alpha=FDR_Q,method="fdr_bh")
    return {c:{"simes_p":pmap[c],"bh_q":float(q),"reject_q_0_05":bool(r)} for c,q,r in zip(valid,qvals,rej)}

family_fdr_outcome=bh_family(family_praw_outcome)
family_fdr_reliability=bh_family(family_praw_reliability)

# Global 32-dimensional BanglaClick delta-RD Wald statistic using the same joint cluster-bootstrap design.
point_vec=np.array([
    next(x["delta_rd"] for x in cue_transport if x["cue"]==cue and x["target"]==f"banglaclick:{plat}")
    for cue in CUE_FAMILIES for plat in ["news","facebook","youtube","twitter"]
],float)

boot_mat=[]
rng_global=np.random.default_rng(BOOTSTRAP_SEED+2)
global_start=time.monotonic()
print(f"[H3 global Wald bootstrap] START — replicates={BOOTSTRAP_REPS}",flush=True)
for rep in range(1,BOOTSTRAP_REPS+1):
    sw=src_h3["plan"].draw_row_counts(rng_global)
    tw={p:bc_h3[p]["plan"].draw_row_counts(rng_global) for p in ["news","facebook","youtube","twitter"]}
    row=[]
    for cue in CUE_FAMILIES:
        sr=_group_gap_1d(src_h3["y"],src_h3["cues"][cue],sw)
        for p in ["news","facebook","youtube","twitter"]:
            tr=_group_gap_1d(bc_h3[p]["y"],bc_h3[p]["cues"][cue],tw[p])
            row.append(tr-sr)
    boot_mat.append(row)

    if rep%100==0:
        _phase_guard("H3 global Wald bootstrap",global_start,rep,BOOTSTRAP_REPS)
    if rep%BOOTSTRAP_PROGRESS_EVERY==0 or rep==BOOTSTRAP_REPS:
        np.save(CHECKPOINT_DIR/"H3_GLOBAL_BOOTSTRAP_PARTIAL.npy",np.asarray(boot_mat,float))
        write_json(
            CHECKPOINT_DIR/"H3_GLOBAL_BOOTSTRAP_CHECKPOINT.json",
            {
                "schema_version":2,
                "completed_replicates":rep,
                "total_replicates":BOOTSTRAP_REPS,
                "rng_state":rng_global.bit_generator.state,
            },
        )
        _progress("H3 global Wald bootstrap",rep,BOOTSTRAP_REPS,global_start)

B=np.asarray(boot_mat,float)
mask=np.isfinite(point_vec)&np.all(np.isfinite(B),axis=0)
pv=point_vec[mask]
if len(pv)==0:
    cov=np.empty((0,0));rank=0;wald=None;global_p=None
else:
    cov=np.atleast_2d(np.cov(B[:,mask],rowvar=False));rank=int(np.linalg.matrix_rank(cov))
    wald=float(pv @ np.linalg.pinv(cov) @ pv) if rank else None
    global_p=float(chi2.sf(wald,rank)) if rank else None
print(f"[H3 global Wald bootstrap] COMPLETE — elapsed={time.monotonic()-global_start:.1f}s",flush=True)

# Independent BaitBuster replication for outcome and cue-conditioned proper-loss transport.
bb_rep=[]
rng_rep=np.random.default_rng(BOOTSTRAP_SEED+3)
bb_start=time.monotonic()
print(f"[H3 BaitBuster replication] START — cues={len(CUE_FAMILIES)}, replicates_per_cue={BOOTSTRAP_REPS}",flush=True)
for cue_i,cue in enumerate(CUE_FAMILIES,1):
    src_rd=_group_gap_1d(src_h3["y"],src_h3["cues"][cue],src_h3["ones"])
    tgt_rd=_group_gap_1d(bb_h3["y"],bb_h3["cues"][cue],bb_h3["ones"])
    src_nll_seed=_group_gap_by_seed(src_h3["loss_by_seed"],src_h3["cues"][cue],src_h3["ones"])
    tgt_nll_seed=_group_gap_by_seed(bb_h3["loss_by_seed"],bb_h3["cues"][cue],bb_h3["ones"])
    by_seed=tgt_nll_seed-src_nll_seed
    point_rd=tgt_rd-src_rd
    point_nll=float(np.nanmean(by_seed))

    rd_boot=[];nll_boot=[]
    detail=f"cue={cue_i}/{len(CUE_FAMILIES)} {cue}"
    print(f"[H3 BaitBuster replication] CUE START — {detail}",flush=True)
    for rep in range(1,BOOTSTRAP_REPS+1):
        sw=src_h3["plan"].draw_row_counts(rng_rep)
        tw=bb_h3["plan"].draw_row_counts(rng_rep)
        rd_boot.append(
            _group_gap_1d(bb_h3["y"],bb_h3["cues"][cue],tw)
            - _group_gap_1d(src_h3["y"],src_h3["cues"][cue],sw)
        )
        sd=_group_gap_by_seed(src_h3["loss_by_seed"],src_h3["cues"][cue],sw)
        td=_group_gap_by_seed(bb_h3["loss_by_seed"],bb_h3["cues"][cue],tw)
        nll_boot.append(float(np.nanmean(td-sd)))

        if rep%100==0:
            _phase_guard("H3 BaitBuster replication",bb_start,rep,BOOTSTRAP_REPS,{"cue":cue})
        if rep%BOOTSTRAP_PROGRESS_EVERY==0 or rep==BOOTSTRAP_REPS:
            _progress("H3 BaitBuster replication",rep,BOOTSTRAP_REPS,bb_start,detail)

    bb_rep.append({
        "cue":cue,
        "delta_rd":point_rd,"delta_rd_ci95":bootstrap_ci(rd_boot),"delta_rd_p":bootstrap_p_two_sided(rd_boot),
        "ebm_delta_nll_gap_mean_5seeds":point_nll,
        "ebm_delta_nll_gap_ci95":bootstrap_ci(nll_boot),
        "ebm_delta_nll_gap_p":bootstrap_p_two_sided(nll_boot),
        "ebm_delta_nll_gap_by_seed":[float(x) if np.isfinite(x) else None for x in by_seed],
    })
    write_json(
        CHECKPOINT_DIR/"H3_BAITBUSTER_CHECKPOINT.json",
        {
            "schema_version":2,
            "completed_cues":cue_i,
            "total_cues":len(CUE_FAMILIES),
            "replicates_per_cue":BOOTSTRAP_REPS,
            "rng_state":rng_rep.bit_generator.state,
            "rows":bb_rep,
            "last_completed_cue":cue,
        },
    )
    print(f"[H3 BaitBuster replication] CUE COMPLETE — {detail}",flush=True)

write_json(OUT_ROOT/"H3_RESULTS.json",{
    "schema_version":2,
    "primary":"prespecified cue–outcome and cue-conditioned proper-loss transport",
    "proper_loss":"NLL on frozen temperature-scaled deployment probabilities",
    "global_banglaclick_outcome_wald":{"statistic":wald,"df":rank,"p":global_p},
    "global_banglaclick_wald":{"statistic":wald,"df":rank,"p":global_p},
    "banglaclick_cue_platform_contrasts":cue_transport,
    "family_fdr_outcome":family_fdr_outcome,
    "family_fdr_reliability":family_fdr_reliability,
    "family_fdr":family_fdr_outcome,
    "baitbuster_replication":bb_rep,
    "causal_language":False,
    "interpretation":"association/transport of predictive outcome and proper-loss relationships under observed distribution shift"
})
print(
    f"H3 observational analysis complete; global outcome-transport p={global_p}; "
    f"elapsed={time.monotonic()-h3_start:.1f}s",
    flush=True,
)


[H3 cue/platform bootstrap] START — contrasts=32, replicates_per_contrast=5000
[H3 cue/platform bootstrap] CONTRAST START — contrast=1/32 cue=curiosity_information_withholding platform=news
[H3 cue/platform bootstrap] 1000/5000 (20.0%) elapsed=0.4s eta=1.4s contrast=1/32 cue=curiosity_information_withholding platform=news
[H3 cue/platform bootstrap] 2000/5000 (40.0%) elapsed=0.7s eta=1.1s contrast=1/32 cue=curiosity_information_withholding platform=news
[H3 cue/platform bootstrap] 3000/5000 (60.0%) elapsed=1.1s eta=0.7s contrast=1/32 cue=curiosity_information_withholding platform=news
[H3 cue/platform bootstrap] 4000/5000 (80.0%) elapsed=1.4s eta=0.4s contrast=1/32 cue=curiosity_information_withholding platform=news
[H3 cue/platform bootstrap] 5000/5000 (100.0%) elapsed=1.8s eta=0.0s contrast=1/32 cue=curiosity_information_withholding platform=news
[H3 cue/platform bootstrap] CONTRAST COMPLETE — contrast=1/32 cue=curiosity_information_withholding platform=news
[H3 cue/platform bootstra

/tmp/ipykernel_23/1730523401.py:98: RuntimeWarning: Mean of empty slice
  nll_boot.append(float(np.nanmean(td-sd)))


[H3 cue/platform bootstrap] 2000/5000 (40.0%) elapsed=4.4s eta=6.7s contrast=3/32 cue=curiosity_information_withholding platform=youtube
[H3 cue/platform bootstrap] 3000/5000 (60.0%) elapsed=4.8s eta=3.2s contrast=3/32 cue=curiosity_information_withholding platform=youtube
[H3 cue/platform bootstrap] 4000/5000 (80.0%) elapsed=5.2s eta=1.3s contrast=3/32 cue=curiosity_information_withholding platform=youtube
[H3 cue/platform bootstrap] 5000/5000 (100.0%) elapsed=5.5s eta=0.0s contrast=3/32 cue=curiosity_information_withholding platform=youtube
[H3 cue/platform bootstrap] CONTRAST COMPLETE — contrast=3/32 cue=curiosity_information_withholding platform=youtube
[H3 cue/platform bootstrap] CONTRAST START — contrast=4/32 cue=curiosity_information_withholding platform=twitter
[H3 cue/platform bootstrap] 1000/5000 (20.0%) elapsed=5.8s eta=23.3s contrast=4/32 cue=curiosity_information_withholding platform=twitter
[H3 cue/platform bootstrap] 2000/5000 (40.0%) elapsed=6.1s eta=9.2s contrast=4/32 

/tmp/ipykernel_23/1730523401.py:218: RuntimeWarning: Mean of empty slice
  nll_boot.append(float(np.nanmean(td-sd)))


[H3 BaitBuster replication] 3000/5000 (60.0%) elapsed=2.5s eta=1.6s cue=1/8 curiosity_information_withholding
[H3 BaitBuster replication] 4000/5000 (80.0%) elapsed=3.3s eta=0.8s cue=1/8 curiosity_information_withholding
[H3 BaitBuster replication] 5000/5000 (100.0%) elapsed=4.2s eta=0.0s cue=1/8 curiosity_information_withholding
[H3 BaitBuster replication] CUE COMPLETE — cue=1/8 curiosity_information_withholding
[H3 BaitBuster replication] CUE START — cue=2/8 direct_address_imperative
[H3 BaitBuster replication] 1000/5000 (20.0%) elapsed=5.0s eta=20.1s cue=2/8 direct_address_imperative
[H3 BaitBuster replication] 2000/5000 (40.0%) elapsed=5.9s eta=8.8s cue=2/8 direct_address_imperative
[H3 BaitBuster replication] 3000/5000 (60.0%) elapsed=6.8s eta=4.5s cue=2/8 direct_address_imperative
[H3 BaitBuster replication] 4000/5000 (80.0%) elapsed=7.6s eta=1.9s cue=2/8 direct_address_imperative
[H3 BaitBuster replication] 5000/5000 (100.0%) elapsed=8.5s eta=0.0s cue=2/8 direct_address_imperativ

In [11]:
# ---------- BanglaClick diagnostic corroboration (model-independent annotations) ----------
# These five fields come directly from the frozen physical workbook. They are supporting
# annotations only and are not statistically independent of the BanglaClick score construction.
def diagnostic_numeric(series, diagnostic_name):
    if diagnostic_name == "diag_fast_fail_count":
        raw = series.astype("string")
        return pd.to_numeric(raw.str.extract(r"^\s*(\d+)", expand=False), errors="coerce").to_numpy(float)
    return pd.to_numeric(series, errors="coerce").to_numpy(float)

actual_diagnostics = [
    "diag_fast_fail_count",
    "diag_fast_level",
    "diag_strong_trigger_count",
    "diag_weak_trigger_count",
    "diag_trigger_level",
]
base_bc = BC_P[(BC_P.model=="ebm") & (BC_P.seed==SEEDS[0])].drop_duplicates("row_id").reset_index(drop=True)
corro = []
for cue in CUE_FAMILIES:
    present = base_bc[cue].to_numpy() > 0
    for d in actual_diagnostics:
        x = diagnostic_numeric(base_bc[d], d)
        ok = np.isfinite(x)
        if ok.sum() == 0 or present[ok].all() or (~present[ok]).all():
            corro.append({
                "cue": cue,
                "diagnostic": d,
                "status": "UNINTERPRETABLE_WITH_FROZEN_DIAGNOSTIC_ENCODING",
                "n_finite": int(ok.sum()),
            })
            continue
        diff = float(np.mean(x[ok & present]) - np.mean(x[ok & ~present]))
        corro.append({
            "cue": cue,
            "diagnostic": d,
            "status": "ESTIMATED_DESCRIPTIVE_ONLY",
            "mean_difference_cue_present_minus_absent": diff,
            "n": int(ok.sum()),
        })
write_json(
    OUT_ROOT/"BANGLACLICK_DIAGNOSTIC_CORROBORATION.json",
    {
        "schema_version": 2,
        "diagnostic_columns": BANGLACLICK_DIAGNOSTIC_COLUMNS,
        "note": (
            "Supporting descriptive criterion evidence only; not statistically independent "
            "of BanglaClick score construction and not part of schema-role selection."
        ),
        "rows": corro,
    },
)


## H3-P — Controlled Cue Neutralization
The perturbation evaluates controlled model sensitivity for the three frozen cue families. It does not identify causal effects on real-world clickbait status.


In [12]:
# ---------- Mandatory minimal H3 controlled neutralization ----------
PERT_FAMS=["curiosity_information_withholding","direct_address_imperative","punctuation_intensity"]

def neutralize(title,fam):
    # Both members of each controlled pair use the same P0-normalized representation, so the only pairwise change is cue deletion.
    text=norm_text(title);orig_tokens=text.split();new=text;spec=CUE_SCHEMA["families"][fam]
    if fam=="punctuation_intensity":
        for p in spec["punctuation"]:new=new.replace(p,"")
        new=re.sub(spec["repeated_pattern"],"",new)
    else:
        for term in sorted(spec.get("terms",[]),key=lambda x:(-len(norm_text(x)),norm_text(x))):
            nt=norm_text(term)
            if " " in nt:new=new.replace(nt," ")
            else:new=re.sub(rf"(?<!\S){re.escape(nt)}(?!\S)"," ",new)
    new=re.sub(r"\s+"," ",new).strip();new_tokens=new.split();removed=len(orig_tokens)-len(new_tokens)
    if len(orig_tokens)==0:return None,"EMPTY_ORIGINAL"
    if removed/len(orig_tokens)>.20:return None,"REMOVAL_GT_20_PERCENT_TOKENS"
    if len(new_tokens)<3:return None,"REMAINS_LT_3_TOKENS"
    if new==text:return None,"NO_CHANGE"
    return new,"INCLUDED"

def logits_for_subset(model,seed,titles):
    rec=model_rec(model,seed);verify_model_rec(rec);p=S3/rec["artifact_path"]
    if model=="ebm":return np.asarray(joblib.load(p).decision_function(cue_features(titles).astype(float)),float)
    if model=="banglabert":return infer_transformer(p,list(titles),int(CONFIG["models"][model]["base_config"]["max_length"]))
    raise ValueError(model)

pert_sources={
    "source_test":src,
    "banglaclick:news":bc[bc.platform=="news"],
    "banglaclick:facebook":bc[bc.platform=="facebook"],
    "banglaclick:youtube":bc[bc.platform=="youtube"],
    "banglaclick:twitter":bc[bc.platform=="twitter"],
    "baitbuster:youtube_human":bb,
}
pair_rows=[];exclusion_rows=[]
print("[H3-P pair construction] START",flush=True)
for dname,df in pert_sources.items():
    cf=cue_features(df.title.tolist())
    for fam in PERT_FAMS:
        for i in np.flatnonzero(cf[fam].to_numpy()>0):
            original_norm=norm_text(df.iloc[i].title);nt,status=neutralize(df.iloc[i].title,fam)
            if status!="INCLUDED":
                exclusion_rows.append({"domain":dname,"row_id":str(df.iloc[i].row_id),"cue":fam,"reason":status});continue
            pair_rows.append({
                "domain":dname,
                "row_id":str(df.iloc[i].row_id),
                "cluster_id":str(df.iloc[i].cluster_id),
                "cue":fam,
                "original_title":original_norm,
                "neutralized_title":nt,
            })
    print(f"[H3-P pair construction] domain={dname} cumulative_pairs={len(pair_rows)} exclusions={len(exclusion_rows)}",flush=True)

PAIR_BASE=pd.DataFrame(pair_rows)
if PAIR_BASE.empty:hard_stop("No eligible H3 perturbation pairs after frozen exclusions")
print(f"[H3-P pair construction] COMPLETE — eligible_pairs={len(PAIR_BASE)} exclusions={len(exclusion_rows)}",flush=True)

# One model load/inference call per model/seed across all controlled pairs.
pert_parts=[]
all_titles=PAIR_BASE.original_title.tolist()+PAIR_BASE.neutralized_title.tolist()
n_pairs=len(PAIR_BASE)
h3p_infer_total=2*len(SEEDS)
h3p_infer_done=0
print(
    f"[H3-P inference] START — model_seed_jobs={h3p_infer_total}, "
    f"titles_per_job={2*n_pairs}",
    flush=True,
)
for model in ["ebm","banglabert"]:
    for seed in SEEDS:
        h3p_infer_done+=1
        job_start=time.monotonic()
        print(
            f"[H3-P inference] JOB START {h3p_infer_done}/{h3p_infer_total} "
            f"model={model} seed={seed}",
            flush=True,
        )
        zz=logits_for_subset(model,seed,all_titles)
        if len(zz)!=2*n_pairs:
            hard_stop("Perturbation inference length mismatch",{"model":model,"seed":seed})
        q=PAIR_BASE.copy()
        q["model"]=model;q["seed"]=seed
        q["delta_z"]=zz[:n_pairs]-zz[n_pairs:]
        pert_parts.append(q)
        q.to_parquet(CHECKPOINT_DIR/f"H3P_INFERENCE_{model}_{seed}.parquet",index=False)
        print(
            f"[H3-P inference] JOB COMPLETE {h3p_infer_done}/{h3p_infer_total} "
            f"model={model} seed={seed} elapsed={time.monotonic()-job_start:.1f}s",
            flush=True,
        )

PERT=pd.concat(pert_parts,ignore_index=True).sort_values(["domain","cue","model","seed","row_id"],kind="stable")
EXCL=pd.DataFrame(exclusion_rows,columns=["domain","row_id","cue","reason"]).sort_values(["domain","cue","row_id"],kind="stable")
PERT.to_parquet(OUT_ROOT/"H3_PERTURBATION_PAIRS.parquet",index=False)
EXCL.to_csv(OUT_ROOT/"H3_PERTURBATION_EXCLUSIONS.csv",index=False)
print(f"[H3-P inference] COMPLETE — perturbation_rows={len(PERT)}",flush=True)

# Cluster-bootstrap mean delta_z and source-to-target difference.
# Cluster membership is precomputed once per group; each draw uses row multiplicities.
pert_results=[]
rngp=np.random.default_rng(BOOTSTRAP_SEED+4)
source_groups={
    (cue,model,int(seed)):sg.reset_index(drop=True)
    for (cue,model,seed),sg in PERT[PERT.domain=="source_test"].groupby(["cue","model","seed"],sort=True)
}
target_group_records=[]
for (cue,model,seed),sg in PERT[PERT.domain=="source_test"].groupby(["cue","model","seed"],sort=True):
    for dname,tg in PERT[
        (PERT.domain!="source_test")&(PERT.cue==cue)&(PERT.model==model)&(PERT.seed==seed)
    ].groupby("domain",sort=True):
        target_group_records.append((cue,model,int(seed),dname,tg.reset_index(drop=True)))

total_pert_contrasts=len(target_group_records)
pert_start=time.monotonic()
print(
    f"[H3-P bootstrap] START — contrasts={total_pert_contrasts}, "
    f"replicates_per_contrast={BOOTSTRAP_REPS}",
    flush=True,
)

plan_cache={}
def _pert_plan(domain,cue,frame):
    key=(domain,cue,tuple(frame.row_id.astype(str).tolist()))
    # key includes the row identity so reuse cannot cross a different pair subset.
    if key not in plan_cache:
        plan_cache[key]=ClusterBootstrapPlan(frame.cluster_id.to_numpy(),f"{domain}|{cue}")
    return plan_cache[key]

for ci,(cue,model,seed,dname,tg) in enumerate(target_group_records,1):
    sg=source_groups[(cue,model,seed)]
    spl=_pert_plan("source_test",cue,sg)
    tpl=_pert_plan(dname,cue,tg)
    sv=sg.delta_z.to_numpy(float);tv=tg.delta_z.to_numpy(float)
    point_s=float(np.mean(sv));point_t=float(np.mean(tv))
    diffs=[]
    detail=f"contrast={ci}/{total_pert_contrasts} cue={cue} model={model} seed={seed} target={dname}"
    print(f"[H3-P bootstrap] CONTRAST START — {detail}",flush=True)

    for rep in range(1,BOOTSTRAP_REPS+1):
        sw=spl.draw_row_counts(rngp);tw=tpl.draw_row_counts(rngp)
        diffs.append(_weighted_mean(tv,tw)-_weighted_mean(sv,sw))

        if rep%100==0:
            _phase_guard("H3-P bootstrap",pert_start,rep,BOOTSTRAP_REPS,{"contrast":detail})
        if rep%BOOTSTRAP_PROGRESS_EVERY==0 or rep==BOOTSTRAP_REPS:
            _progress("H3-P bootstrap",rep,BOOTSTRAP_REPS,pert_start,detail)

    pert_results.append({
        "cue":cue,"model":model,"seed":int(seed),"target":dname,
        "source_mean_delta_z":point_s,
        "target_mean_delta_z":point_t,
        "target_minus_source_delta_z":point_t-point_s,
        "ci95":bootstrap_ci(diffs),
        "p":bootstrap_p_two_sided(diffs),
    })
    write_json(
        CHECKPOINT_DIR/"H3P_BOOTSTRAP_CHECKPOINT.json",
        {
            "schema_version":2,
            "completed_contrasts":ci,
            "total_contrasts":total_pert_contrasts,
            "replicates_per_contrast":BOOTSTRAP_REPS,
            "rng_state":rngp.bit_generator.state,
            "rows":pert_results,
            "last_completed":{
                "cue":cue,"model":model,"seed":int(seed),"target":dname
            },
        },
    )
    print(f"[H3-P bootstrap] CONTRAST COMPLETE — {detail}",flush=True)

write_json(
    OUT_ROOT/"H3_PERTURBATION_RESULTS.json",
    {
        "schema_version":1,
        "response":"delta_z = normalized-original logit - cue-neutralized logit",
        "causal_real_world_claim":False,
        "pair_representation":"Both original and neutralized inputs share P0 normalization; only frozen cue span/punctuation is removed.",
        "models":["ebm","banglabert"],
        "families":PERT_FAMS,
        "rows":pert_results,
        "pairs_sha256":sha256_file(OUT_ROOT/"H3_PERTURBATION_PAIRS.parquet"),
        "exclusions_sha256":sha256_file(OUT_ROOT/"H3_PERTURBATION_EXCLUSIONS.csv"),
    },
)
print(
    f"H3 perturbation complete — contrasts={len(pert_results)} "
    f"bootstrap_elapsed={time.monotonic()-pert_start:.1f}s",
    flush=True,
)


[H3-P pair construction] START
[H3-P pair construction] domain=source_test cumulative_pairs=669 exclusions=14
[H3-P pair construction] domain=banglaclick:news cumulative_pairs=996 exclusions=24
[H3-P pair construction] domain=banglaclick:facebook cumulative_pairs=1565 exclusions=91
[H3-P pair construction] domain=banglaclick:youtube cumulative_pairs=2388 exclusions=91
[H3-P pair construction] domain=banglaclick:twitter cumulative_pairs=2522 exclusions=91
[H3-P pair construction] domain=baitbuster:youtube_human cumulative_pairs=8139 exclusions=99
[H3-P pair construction] COMPLETE — eligible_pairs=8139 exclusions=99
[H3-P inference] START — model_seed_jobs=10, titles_per_job=16278
[H3-P inference] JOB START 1/10 model=ebm seed=42
[H3-P inference] JOB COMPLETE 1/10 model=ebm seed=42 elapsed=6.5s
[H3-P inference] JOB START 2/10 model=ebm seed=137
[H3-P inference] JOB COMPLETE 2/10 model=ebm seed=137 elapsed=6.5s
[H3-P inference] JOB START 3/10 model=ebm seed=2718
[H3-P inference] JOB COMPL

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[H3-P inference] JOB COMPLETE 6/10 model=banglabert seed=42 elapsed=45.5s
[H3-P inference] JOB START 7/10 model=banglabert seed=137


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[H3-P inference] JOB COMPLETE 7/10 model=banglabert seed=137 elapsed=43.1s
[H3-P inference] JOB START 8/10 model=banglabert seed=2718


...

[H3-P inference] JOB COMPLETE 8/10 model=banglabert seed=2718 elapsed=44.6s
[H3-P inference] JOB START 9/10 model=banglabert seed=31415


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[H3-P inference] JOB COMPLETE 9/10 model=banglabert seed=31415 elapsed=43.6s
[H3-P inference] JOB START 10/10 model=banglabert seed=65537


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[H3-P inference] JOB COMPLETE 10/10 model=banglabert seed=65537 elapsed=44.4s
[H3-P inference] COMPLETE — perturbation_rows=81390
[H3-P bootstrap] START — contrasts=150, replicates_per_contrast=5000
[H3-P bootstrap] CONTRAST START — contrast=1/150 cue=curiosity_information_withholding model=banglabert seed=42 target=baitbuster:youtube_human
[H3-P bootstrap] 1000/5000 (20.0%) elapsed=0.1s eta=0.2s contrast=1/150 cue=curiosity_information_withholding model=banglabert seed=42 target=baitbuster:youtube_human
[H3-P bootstrap] 2000/5000 (40.0%) elapsed=0.1s eta=0.2s contrast=1/150 cue=curiosity_information_withholding model=banglabert seed=42 target=baitbuster:youtube_human
[H3-P bootstrap] 3000/5000 (60.0%) elapsed=0.2s eta=0.1s contrast=1/150 cue=curiosity_information_withholding model=banglabert seed=42 target=baitbuster:youtube_human
[H3-P bootstrap] 4000/5000 (80.0%) elapsed=0.2s eta=0.1s contrast=1/150 cue=curiosity_information_withholding model=banglabert seed=42 target=baitbuster:you

## Statistical Analysis
The locked analysis uses five seeds, 5,000 cluster-bootstrap attempts, 95% percentile intervals, Simes aggregation within cue families, and separate BH-FDR control over the eight prespecified families for outcome and proper-loss transport. Effective finite/two-class bootstrap counts are recorded where resamples are non-estimable.

## Sensitivity and Results
The locked bootstrap count, clustered resampling, BH-FDR control, prevalence/structural sensitivity analyses, and five-seed reporting are preserved.


In [13]:
print("[Sensitivity/statistical summary] START",flush=True)
# ---------- Sensitivity + statistical summary ----------
# Save metric sensitivity records already computed: source-prior and structural-composition standardization.
write_json(OUT_ROOT/"SENSITIVITY_RESULTS.json",{"schema_version":1,"source_positive_prevalence":source_pi,
                                                "methods":["source-prior-standardized class weighting","source-feature structural composition weighting on token count + code mixing"],
                                                "metric_rows":metric_rows})
write_json(OUT_ROOT/"STATISTICAL_SUMMARY.json",{"schema_version":1,"bootstrap_replicates":BOOTSTRAP_REPS,"ci":"95% percentile","bootstrap_seed":BOOTSTRAP_SEED,
                                                "cluster_rule":"strongest available source entity; BaitBuster=channel_id; source TEST=row; BanglaClick=SOURCE_URL",
                                                "h1_h2_bootstrap":stat_rows,"h3_global":json.loads((OUT_ROOT/"H3_RESULTS.json").read_text(encoding="utf-8"))["global_banglaclick_wald"],
                                                "h3_family_fdr_outcome":json.loads((OUT_ROOT/"H3_RESULTS.json").read_text(encoding="utf-8"))["family_fdr_outcome"],
                                                "h3_family_fdr_reliability":json.loads((OUT_ROOT/"H3_RESULTS.json").read_text(encoding="utf-8"))["family_fdr_reliability"],
                                                "fdr_method":"Separate Benjamini-Hochberg control over 8 prespecified cue-family Simes p-values for outcome transport and proper-loss transport","fdr_q":FDR_Q})

# Explicit falsifiable support screens; these summarize evidence but never abort execution on scientific falsification.
stat_lookup={(r["domain"],r["model"],int(r["seed"])):r for r in stat_rows}
metric_lookup={(r["domain"],r["model"],int(r["seed"])):r for r in metric_rows}
h1_assess=[];h2_assess=[]
for r in h1_screen:
    key=(r["domain"],r["model"],int(r["seed"]));st=stat_lookup[key];srcm=metric_lookup[("source_test",r["model"],int(r["seed"]))];tgtm=metric_lookup[key]
    si,ss=srcm["calibrated"]["calibration_intercept"],srcm["calibrated"]["calibration_slope"]
    ti,ts=tgtm["calibrated"]["calibration_intercept"],tgtm["calibrated"]["calibration_slope"]
    cal_away=False
    if None not in (si,ss,ti,ts):cal_away=(abs(ti)>abs(si)) or (abs(ts-1)>abs(ss-1))
    nll_evidence=(r["nll_relative_change"]>=.10 and st["nll_relative_change_ci95"][0] is not None and st["nll_relative_change_ci95"][0]>0)
    bbs_evidence=(r["balanced_brier_relative_change"]>=.10 and st["balanced_brier_relative_change_ci95"][0] is not None and st["balanced_brier_relative_change_ci95"][0]>0)
    supported=bool(r["target_auroc"]>=.70 and r["auroc_drop"]<=.05 and (nll_evidence or bbs_evidence) and cal_away)
    h1_assess.append({**r,"nll_bootstrap_ci95":st["nll_relative_change_ci95"],"balanced_brier_bootstrap_ci95":st["balanced_brier_relative_change_ci95"],
                      "calibration_moved_further_from_ideal":cal_away,"setting_supported":supported})
for r in h2_screen:
    key=(r["domain"],r["model"],int(r["seed"]));st=stat_lookup[key]
    source_positive=(r["source_nll_gain"]>0 and st["h2_source_gain_ci95"][0] is not None and st["h2_source_gain_ci95"][0]>0)
    attenuated=(r["target_minus_source_gain"]<0 and st["h2_target_minus_source_gain_ci95"][1] is not None and st["h2_target_minus_source_gain_ci95"][1]<0)
    h2_assess.append({**r,"source_gain_ci95":st["h2_source_gain_ci95"],"target_gain_ci95":st["h2_target_gain_ci95"],
                      "target_minus_source_gain_ci95":st["h2_target_minus_source_gain_ci95"],"setting_supported":bool(source_positive and attenuated)})

def domain_model_consensus(rows):
    out=[]
    for (d,m),g in pd.DataFrame(rows).groupby(["domain","model"],sort=True):
        n=int(g.setting_supported.sum());out.append({"domain":d,"model":m,"supporting_seeds":n,"total_seeds":int(len(g)),"consensus_ge_4_of_5":bool(n>=4)})
    return out
h1_json=json.loads((OUT_ROOT/"H1_RESULTS.json").read_text(encoding="utf-8"));h1_json["confirmatory_assessment_rows"]=h1_assess;h1_json["domain_model_seed_consensus"]=domain_model_consensus(h1_assess);h1_json["scientific_non_support_is_not_pipeline_failure"]=True;write_json(OUT_ROOT/"H1_RESULTS.json",h1_json)
h2_json=json.loads((OUT_ROOT/"H2_RESULTS.json").read_text(encoding="utf-8"));h2_json["confirmatory_assessment_rows"]=h2_assess;h2_json["domain_model_seed_consensus"]=domain_model_consensus(h2_assess);h2_json["scientific_non_support_is_not_pipeline_failure"]=True;write_json(OUT_ROOT/"H2_RESULTS.json",h2_json)
print("[Sensitivity/statistical summary] COMPLETE",flush=True)


[Sensitivity/statistical summary] START
[Sensitivity/statistical summary] COMPLETE


In [14]:
# ---------- Human-readable tables/figures ----------
import matplotlib.pyplot as plt
figdir=OUT_ROOT/"figures"; tabdir=OUT_ROOT/"tables";figdir.mkdir(exist_ok=True);tabdir.mkdir(exist_ok=True)
# Main metric table (seed-level; no model-winner framing).
flat=[]
for r in metric_rows:
    m=r["calibrated"];flat.append({"domain":r["domain"],"model":r["model"],"seed":r["seed"],"n":r["n"],"prevalence":r["prevalence"],**m})
pd.DataFrame(flat).to_csv(tabdir/"H1_CALIBRATED_METRICS_BY_SEED.csv",index=False)
pd.DataFrame(cue_transport).to_csv(tabdir/"H3_CUE_TRANSPORT_CONTRASTS.csv",index=False)
pd.DataFrame(pert_results).to_csv(tabdir/"H3_PERTURBATION_CONTRASTS.csv",index=False)

# Compact cue-transport figure using EBM-agnostic outcome RD deltas.
plotdf=pd.DataFrame(cue_transport)
for plat in ["news","facebook","youtube","twitter"]:
    d=plotdf[plotdf.target==f"banglaclick:{plat}"]
    fig,ax=plt.subplots(figsize=(9,5));ax.barh(d.cue,d.delta_rd);ax.axvline(0,linewidth=1);ax.set_xlabel("Target − source cue risk-difference");ax.set_title(f"BanglaClick {plat}: prespecified cue transport")
    fig.tight_layout();fig.savefig(figdir/f"h3_cue_transport_{plat}.png",dpi=180);plt.close(fig)
print("Tables/figures written")


Tables/figures written


## Output Artifacts
Predictions, schema audit, H1/H2/H3/H3-P results, perturbation pairs/exclusions, sensitivity results, statistical summaries, tables/figures, runtime metadata, manifests, and checksums are persisted.

## Stage Decision
All confirmatory outputs are persisted and hashed before `PASS`. Scientific non-support is recorded as a result and is never treated as pipeline failure.


In [15]:
# ---------- Stage-05 immutable manifest and PASS ----------
required=["ENVIRONMENT_PREFLIGHT.json","FROZEN_INPUT_DISCOVERY.json","BANGLACLICK_SCHEMA_PROBE.json","BANGLACLICK_SCHEMA_REPORT.json","BANGLACLICK_DIAGNOSTIC_CORROBORATION.json","LOCKED_SOURCE_TEST_PREDICTIONS.parquet","BANGLACLICK_PLATFORM_PREDICTIONS.parquet","BAITBUSTER_HUMAN_PREDICTIONS.parquet",
          "H1_RESULTS.json","H2_RESULTS.json","H3_RESULTS.json","H3_PERTURBATION_RESULTS.json","H3_PERTURBATION_PAIRS.parquet","H3_PERTURBATION_EXCLUSIONS.csv",
          "SENSITIVITY_RESULTS.json","STATISTICAL_SUMMARY.json","EXTERNAL_DATA_ADAPTER_REPORT.json"]
missing=[x for x in required if not (OUT_ROOT/x).is_file()]
if missing:hard_stop("Stage-05 artifact contract incomplete",{"missing":missing})
GPU_RUNTIME={"cuda_available":bool(torch.cuda.is_available()),"device_name":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
             "device_count":int(torch.cuda.device_count()) if torch.cuda.is_available() else 0}
RUN={"schema_version":1,"stage_id":STAGE_ID,"execution_utc":utc_now(),"python":sys.version,"platform":platform.platform(),
     "runtime":{"cpu_count":os.cpu_count(),"gpu":GPU_RUNTIME},
     "packages":{n:pkg(n) for n in ["pandas","numpy","pyarrow","openpyxl","scikit-learn","interpret-core","transformers","torch","scipy","statsmodels","matplotlib","joblib"]},
     "seeds":SEEDS,"bootstrap_reps":BOOTSTRAP_REPS,"git":git_state(),"input_hashes":{"stage03_checksums":S3_SUM,"stage04_checksums":S4_SUM,"canonical_zip_sha256_provenance":EXPECTED_ZIP_SHA256,"frozen_content_identity":FROZEN_PACKAGE_IDENTITY,"frozen_input_mode":FROZEN_INPUT_MODE,"banglaclick_workbook":BANGLACLICK_SHA256,"banglaclick_commit":BANGLACLICK_COMMIT,"baitbuster":BAITBUSTER_SHA256},
     "scientific_hierarchy":"H3 primary; H1/H2 supporting; H4 deferred","target_tuning":False}
write_json(OUT_ROOT/"RUN_METADATA.json",RUN)
h3_now=json.loads((OUT_ROOT/"H3_RESULTS.json").read_text(encoding="utf-8"))
write_text(OUT_ROOT/"STAGE05_SUMMARY.md",
           "# Stage 05 execution summary\n\n"
           f"- Source TEST predictions: {len(SRC_P)} model-seed rows\n"
           f"- BanglaClick predictions: {len(BC_P)} model-seed rows\n"
           f"- BaitBuster-human predictions: {len(BB_P)} model-seed rows\n"
           f"- H3 global BanglaClick transport p-value: {h3_now['global_banglaclick_wald'].get('p')}\n"
           "- Scientific hierarchy: H3 primary; H1/H2 supporting; H4 deferred to Stage 06.\n"
           "- No external tuning, recalibration, threshold selection, or cue selection occurred.\n")
manifest={"schema_version":1,"stage_id":STAGE_ID,"protocol_version":PROTOCOL_VERSION,"upstream_stage03_checksum_manifest_sha256":S3_SUM,"upstream_stage04_checksum_manifest_sha256":S4_SUM,
          "canonical_zip_sha256_provenance":EXPECTED_ZIP_SHA256,"frozen_content_identity_sha256":FROZEN_PACKAGE_IDENTITY,"banglaclick_workbook_sha256":BANGLACLICK_SHA256,"banglaclick_commit":BANGLACLICK_COMMIT,"frozen_input_mode":FROZEN_INPUT_MODE,"artifacts":{x:sha256_file(OUT_ROOT/x) for x in required},"stage05_summary_sha256":sha256_file(OUT_ROOT/"STAGE05_SUMMARY.md"),"run_metadata_sha256":sha256_file(OUT_ROOT/"RUN_METADATA.json")}
manifest["manifest_payload_sha256"]=hashlib.sha256(cjb(manifest)).hexdigest();write_json(OUT_ROOT/"STAGE05_ARTIFACT_MANIFEST.json",manifest)
entries=[]
for p in sorted(OUT_ROOT.rglob("*"),key=str):
    if p.is_file() and p.name not in {"STAGE05_SHA256SUMS.txt","STAGE_STATUS.json"}:entries.append((str(p.relative_to(OUT_ROOT)).replace("\\","/"),sha256_file(p)))
write_text(OUT_ROOT/"STAGE05_SHA256SUMS.txt","\n".join(f"{h}  {r}" for r,h in entries)+"\n")
set_status("PASS","Locked source/external H1/H2/H3/H3-P evaluation completed",{"output_root":str(OUT_ROOT),"checksum_entries":len(entries)})
print("STAGE 05 COMPLETE — PASS")


STAGE 05 STATUS: PASS — Locked source/external H1/H2/H3/H3-P evaluation completed
STAGE 05 COMPLETE — PASS
